# ILC vs NST on TCGA-BRCA

Explainable classification of invasive lobular carcinoma (ILC) vs carcinoma of no special type (NST) on TCGA-BRCA diagnostic slides, from UNI2-h tile embeddings.

Setup: `pip install -r requirements.txt`, `python download.py` (needs Hugging Face access to UNI2-h and CONCH), then start Jupyter from the repo folder.

Each step loads its result from disk if it is there and computes it otherwise. Delete a step's files to recompute it.

In [1]:
import concurrent.futures
import contextlib
import functools
import hashlib
import itertools
import json
import math
import os
import threading
import time
import urllib.request
import zipfile
from io import BytesIO

import h5py
import numpy as np
import openslide
import pandas as pd
import timm
import torch
import torchvision
from conch.open_clip_custom import create_model_from_pretrained, get_tokenizer
from IPython.display import Image as IPythonImage
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.figure import Figure
from matplotlib.image import imread, imsave
from matplotlib.path import Path
from PIL import Image
from scipy.ndimage import gaussian_filter
from scipy.ndimage import label as scipy_label
from scipy.spatial.distance import cdist
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from timm.layers import LayerNorm as TimmLayerNorm
from timm.layers.attention import Attention as TimmAttention
from timm.layers.mlp import GluMlp
from torch import nn
from torch.autograd import Function
from torchvision import transforms
from torchvision.utils import make_grid
from zennit.core import BasicHook, Composite, stabilize
from zennit.rules import Epsilon, Gamma

# Jupyter starts in the notebook's folder, the repo root
root = os.path.abspath("")
if not os.path.isfile(os.path.join(root, "download.py")):
    raise RuntimeError(f"start Jupyter from the ilc_tcga folder, not {root}")
results_dir = os.path.join(root, "results")
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

/Users/jona/miniforge3/envs/ilcTcga/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
def load_or_run(name, run, outputs=()):
    # prints results/<name>.txt, running the step into it first unless that file or all the step's outputs exist
    path = os.path.join(results_dir, name + ".txt")
    if not os.path.exists(path) and outputs and all(os.path.exists(output) for output in outputs):
        print("on disk, not recomputed:\n" + "\n".join(outputs))
        return
    if not os.path.exists(path):
        os.makedirs(results_dir, exist_ok=True)
        with open(path + ".incomplete", "w") as f, contextlib.redirect_stdout(f):
            run()
        os.rename(path + ".incomplete", path)
    print(open(path).read(), end="")

# display with reduced resolution
def show(path, width=800):
    rgb = Image.open(path).convert("RGB")
    rgb.thumbnail((width, rgb.height))
    buffer = BytesIO()
    rgb.save(buffer, format="JPEG", quality=80)
    display(IPythonImage(data=buffer.getvalue(), format="jpeg"))

## 1. Metadata and labels

Fetches the TCGA-BRCA diagnostic slides from the GDC API and the Biotab clinical table. ILC = Biotab type ILC **and** ICD-O-3 8520/3, NST = IDC **and** 8500/3, everything else excluded. The 2026-11-02 run gave 187 ILC and 752 NST patients.

In [ ]:
# load labels for the datasets
api = "https://api.gdc.cancer.gov"
meta_dir = os.path.join(root, "data/meta")
biotab_id, biotab_md5 = "8162d394-8b64-4da2-9f5b-d164c54b9608", "f271f56d44b62d41bd6ddb0d26e525b4"
# Biotab histological type and ICD-O-3 code must agree on the type; every other slide is excluded
label_rules = {"Infiltrating Lobular Carcinoma 8520/3": "ILC", "Infiltrating Ductal Carcinoma 8500/3": "NST"}


def run_labels():
    filters = {"cases.project.project_id": "TCGA-BRCA", "data_type": "Slide Image",
               "experimental_strategy": "Diagnostic Slide"}
    content = [{"op": "in", "content": {"field": field, "value": [value]}} for field, value in filters.items()]
    body = {"filters": {"op": "and", "content": content}, "fields": "file_id,file_name,md5sum", "size": 10000}
    request = urllib.request.Request(f"{api}/files", json.dumps(body).encode(), {"Content-Type": "application/json"})
    with urllib.request.urlopen(request) as response, open(os.path.join(meta_dir, "files_slides.json"), "wb") as f:
        f.write(raw := response.read())
    data = json.loads(raw)["data"]
    if data["pagination"]["total"] != len(data["hits"]):
        raise RuntimeError(f"size cut the answer to {len(data['hits'])} of {data['pagination']['total']} slides")
    raw = urllib.request.urlopen(f"{api}/data/{biotab_id}").read()
    if hashlib.md5(raw).hexdigest() != biotab_md5:
        raise ValueError("md5 mismatch for the Biotab patient table")
    # rows 2-3 repeat the column names; a slide whose patient is not in the table raises at .loc
    clinical = pd.read_csv(BytesIO(raw), sep="\t", skiprows=[1, 2], dtype=str).set_index("bcr_patient_barcode")
    # sample code 06 is a metastasis, not the primary tumour the label describes
    slides = pd.DataFrame([hit for hit in data["hits"] if hit["file_name"][13:15] != "06"]).sort_values("file_name")
    slides = slides.assign(patient=slides.file_name.str[:12], site=slides.file_name.str[5:7])
    type_code = clinical.loc[slides.patient, ["histological_type", "icd_o_3_histology"]].agg(" ".join, axis=1)
    slides["label"] = type_code.map(label_rules).fillna("excluded").to_numpy()
    slides.to_csv(os.path.join(meta_dir, "labels.csv"), index=False)
    print(slides.groupby("label").agg(slides=("file_id", "size"), patients=("patient", "nunique")))

In [ ]:
load_or_run("labels", run_labels, [os.path.join(meta_dir, "labels.csv")])

## 2. Train/val/test split

Needs the UNI2-h `.h5` tile features from `download.py`; 7 NST patients from site OL have none. Each tissue source site goes whole into one split, as TCGA sites carry their own staining and scanning signature (Howard et al. 2021). Sites go largest first to the split that brings ILC and NST closest to 70/15/15%; BH alone has 131 patients, so val is larger than 15%.

In [ ]:
labels_csv = os.path.join(root, "data/meta/labels.csv")
h5_dir = os.path.join(root, "data/uni2/h5")
splits_csv = os.path.join(root, "data/meta/splits.csv")
fractions = {"train": 0.7, "val": 0.15, "test": 0.15}
# NST patients from site OL, one slide each, that have no UNI2 features
no_features = {"TCGA-OL-A5RU", "TCGA-OL-A5RV", "TCGA-OL-A5RW", "TCGA-OL-A5RX", "TCGA-OL-A5RY", "TCGA-OL-A5RZ",
               "TCGA-OL-A5S0"}


# split by hospital, so val and test share no site with train
def run_splits():
    slides = pd.read_csv(labels_csv).query("label != 'excluded'")
    slides = slides.assign(h5=slides.file_name.str.replace(".svs", ".h5"))
    has_h5 = slides.h5.map(lambda name: os.path.isfile(os.path.join(h5_dir, name)))
    if set(slides.patient[~has_h5]) != no_features:
        raise ValueError(f"patients without .h5: {sorted(set(slides.patient[~has_h5]))}")
    patients = slides[has_h5].drop_duplicates("patient")
    counts = pd.crosstab(patients.site, patients.label)
    target = np.outer(list(fractions.values()), counts.sum())
    filled = np.zeros_like(target)
    split = {}
    # whole sites, largest first, each to the split it brings closest to its ILC and NST targets; the distance is
    # relative to each target, so ILC weighs as much as NST
    for site, n in counts.loc[sorted(counts.index, key=lambda site: (-counts.loc[site].sum(), site))].iterrows():
        gain = (((filled + n.to_numpy()) / target - 1) ** 2 - (filled / target - 1) ** 2).sum(1)
        split[site] = list(fractions)[gain.argmin()]
        filled[gain.argmin()] += n.to_numpy()
    slides = slides[has_h5].assign(split=slides.site.map(split))
    slides[["patient", "site", "label", "split", "h5"]].to_csv(splits_csv, index=False)
    patients = slides.drop_duplicates("patient")
    print(pd.crosstab(patients.split, patients.label).assign(sites=patients.groupby("split").site.unique()))


load_or_run("splits", run_splits, [splits_csv])

## 3. Tile-level classifier ⏱

The first run keeps the tiles whose centre lies inside a Kather tumour outline of their slide in `data/uni2/tumour_tiles.h5` (25.8 GB; derived from UNI2-h, so git-ignored and never redistributed); 13 patients have no outline and are left out. One linear layer on each tile's embedding, trained with the patient's label on every tile; no MIL, no attention. About 30 min on `mps`.

The 2026-11-07 run reached val AUROC 0.980 and test 0.954. Test informs no choice.

In [ ]:
kather_zip = os.path.join(root, "data/external/kather2021/BRCA.zip")
positive = "ILC"
tumour_cache = os.path.join(root, "data/uni2/tumour_tiles.h5")
checkpoint = os.path.join(root, "models/tile_classifier.pt")
# of the starting weights and the patient order; every later step uses the same seed
seed = 0


# the slide's Kather tumour outlines: one (n, 2) array of level-0 x, y per polygon, none for an empty file
def outlines(archive, slide):
    text = archive.read(f"BRCA/{slide}.csv").decode()
    return [np.array([line.split(",") for line in part.strip().splitlines()], dtype=float)
            for part in text.split("X_base, Y_base") if part.strip()]


# a logit per tile from its UNI2-h embedding; the patient logit is their mean, so tile i's exact share is logit i / n
class TileClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Linear(1536, 1)

    def forward(self, tiles):
        tile_logits = self.head(tiles).squeeze(-1)
        return tile_logits.mean(), tile_logits


# one group per patient with the features, coords and slide of their tumour tiles; patients without any get none
def build_cache(slides):
    with zipfile.ZipFile(kather_zip) as archive, h5py.File(tumour_cache + ".incomplete", "w") as cache:
        for patient, rows in slides.groupby("patient"):
            parts = []
            for position, name in enumerate(rows.h5):
                with h5py.File(os.path.join(h5_dir, name)) as f:
                    features, coords = f["features"][0], f["coords"][0]
                    # the tile size in level-0 px differs per file: 256 on 20x scans up to 1024 on 80x
                    centres = coords + f["coords"].attrs["patch_size_level0"] / 2
                tumour = np.zeros(len(features), bool)
                for polygon in outlines(archive, name[:-len(".h5")]):
                    tumour |= Path(polygon).contains_points(centres)
                parts.append([features[tumour], coords[tumour], np.full(tumour.sum(), position, dtype=np.int32)])
            features, coords, slide = (np.concatenate(part) for part in zip(*parts))
            if not np.isfinite(features).all():
                raise ValueError(f"non-finite features for {patient}")
            if len(features):
                group = cache.create_group(patient)
                # slide: the position of each tile's slide in the attribute "slides"
                group["features"], group["coords"], group["slide"] = features, coords, slide
                group.attrs["slides"] = rows.h5.tolist()
    # written under another name first, so a present cache is a finished one
    os.rename(tumour_cache + ".incomplete", tumour_cache)


def patient_logits(model, patients, cache, device):
    model.eval()
    with torch.no_grad():
        return torch.stack([model(torch.from_numpy(cache[patient]["features"][()]).to(device))[0]
                            for patient in patients.patient])


# every tile inherits its patient's label; AdamW, one step per patient; early stopping on the val loss of the patient
# logit, the quantity we report, with a patience of 10 epochs
def train_model(model, patients, cache, device):
    rng = np.random.default_rng(seed)
    train, val = (patients[patients.split == name] for name in ["train", "val"])
    train_y, val_y = ((part.label == positive).to_numpy(np.float32) for part in [train, val])
    # ILC weighs as much as NST overall
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - train_y).sum() / train_y.sum(), device=device))
    optimiser = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    best_loss, best_epoch = np.inf, None
    for epoch in range(100):
        start, train_loss = time.time(), 0.0
        model.train()
        for index in rng.permutation(len(train)):
            _, tile_logits = model(torch.from_numpy(cache[train.patient.iloc[index]]["features"][()]).to(device))
            # a mean over the patient's tiles, so every patient weighs the same whatever their number of tiles
            loss = loss_fn(tile_logits, torch.tensor(train_y[index], device=device).expand_as(tile_logits))
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            train_loss += loss.item() / len(train)
        val_logits = patient_logits(model, val, cache, device)
        val_loss = loss_fn(val_logits, torch.from_numpy(val_y).to(device)).item()
        if not np.isfinite(val_loss):
            raise ValueError(f"non-finite val loss in epoch {epoch}")
        print(f"epoch {epoch}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val AUROC "
              f"{roc_auc_score(val_y, val_logits.cpu().numpy()):.3f}, {time.time() - start:.0f} s", flush=True)
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best_state = {name: weight.clone() for name, weight in model.state_dict().items()}
        elif epoch - best_epoch >= 10:
            break
    model.load_state_dict(best_state)
    print(f"best epoch {best_epoch}: val loss {best_loss:.4f}")


def run_tile_classifier():
    slides = pd.read_csv(splits_csv)
    if not os.path.exists(tumour_cache):
        build_cache(slides)
    with h5py.File(tumour_cache) as cache:
        patients = slides.drop_duplicates("patient")
        patients = patients[patients.patient.isin(list(cache))]
        torch.manual_seed(seed)
        model = TileClassifier().to(device)
        train_model(model, patients, cache, device)
        torch.save(model.state_dict(), checkpoint)
        logits = patient_logits(model, patients, cache, device).cpu().numpy()
    rows = []
    for name in ["val", "test"]:
        part = (patients.split == name).to_numpy()
        y = (patients.label[part] == positive).to_numpy()
        rows.append({"split": name, "ILC": y.sum(), "NST": (~y).sum(), "auroc": roc_auc_score(y, logits[part]),
                     "balanced_acc": balanced_accuracy_score(y, logits[part] > 0)})
    print(pd.DataFrame(rows).set_index("split").round(3).to_string())


load_or_run("tile_classifier", run_tile_classifier, [tumour_cache, checkpoint])

## 4. Looking at the tile logits

Val tiles only; nothing here trains. Between -4 and 4 the share of ILC tiles per logit bin follows the sigmoid, so tile logits behave like log-odds.

In [ ]:
def run_logit_scale():
    # how tile logits are distributed on val, and whether their magnitude behaves like log-odds
    state = torch.load(checkpoint, map_location="cpu")
    w, b = state["head.weight"].numpy()[0], state["head.bias"].item()
    print(f"bias {b:.3f}, weight norm {np.linalg.norm(w):.3f}")
    patients = pd.read_csv(splits_csv).drop_duplicates("patient").query("split == 'val'")
    with h5py.File(tumour_cache) as cache:
        patients = patients[patients.patient.isin(list(cache))]
        # one patient's features in memory at a time
        logits, norms = zip(*[(x @ w + b, np.linalg.norm(x, axis=1))
                              for x in (cache[patient]["features"][()] for patient in patients.patient)])
    is_ilc = (patients.label == positive).to_numpy()
    n = np.array([len(t) for t in logits])
    rows = pd.DataFrame({"ilc": is_ilc, "n": n, "mean": [t.mean() for t in logits], "std": [t.std() for t in logits],
                         "positive": [(t > 0).mean() for t in logits]})
    t, norms, ilc = np.concatenate(logits), np.concatenate(norms), np.repeat(is_ilc, n)
    # every patient weighs the same and both classes weigh the same, as in the training loss
    weight = np.repeat(1 / n / np.where(is_ilc, is_ilc.sum(), (~is_ilc).sum()), n)
    print(f"{len(rows)} val patients ({is_ilc.sum()} ILC), {len(t)} tiles")
    print("\ntile logit percentiles, tiles of each class pooled")
    for name, part in [("ILC", ilc), ("NST", ~ilc)]:
        print(name, np.round(np.percentile(t[part], [1, 5, 25, 50, 75, 95, 99]), 2))
    print("\nper patient, median over patients")
    print(rows.groupby("ilc")[["n", "mean", "std", "positive"]].median().round(3).to_string())
    print("\nspread between patients vs inside one patient")
    print(f"std of patient means {rows['mean'].std():.3f}, median std inside a patient {rows['std'].median():.3f}")
    print("\nreliability: share of ILC among tiles per logit bin, weighted as in the loss, vs sigmoid of the bin's mean logit")
    edges = [-np.inf, -8, -6, -4, -3, -2, -1, 0, 1, 2, 3, 4, 6, 8, np.inf]
    bins = np.digitize(t, edges)
    table = []
    for index in np.unique(bins):
        part = bins == index
        table.append({"from": edges[index - 1], "to": edges[index], "tiles": part.sum(),
                      "share of weight": weight[part].sum() / weight.sum(),
                      "share ILC": weight[part & ilc].sum() / weight[part].sum(),
                      "sigmoid": 1 / (1 + np.exp(-np.average(t[part], weights=weight[part])))})
    print(pd.DataFrame(table).round(3).to_string(index=False))
    print(f"\nPearson correlation of tile logit and embedding norm: {np.corrcoef(t, norms)[0, 1]:.3f}")
    print(f"logit variance explained by the bias-free projection on the unit weight: "
          f"std of t {t.std():.3f}, mean norm {norms.mean():.2f}, std of norm {norms.std():.2f}")


load_or_run("logit_scale", run_logit_scale)

**Inside vs outside the outlines.** Scores every val tile, also those outside the outlines the model trained on, by distance from the nearest outline. Needs the `.h5` files. The AUROC falls with distance but stays above 0.5; what carries the signal far from the tumour is open (tumour no outline covers, stain or site, the surrounding tissue). Val holds only 2 sites.

In [ ]:
# distance from the nearest outline in tile widths, set before seeing results; at a nominal 0.5 um per 20x px a tile is
# 128 um wide, so about 0.5 and 2 mm
bands = {"under 4": (0, 4), "4 to 16": (4, 16), "over 16": (16, np.inf)}
# a mean over few tiles is noisy, and noise alone would pull an AUROC towards 0.5 and look reassuring
min_tiles = 20


# one patient's tile logits inside and outside the outlines, pooled over the slides as in training, and the distance
# of the tiles outside from the nearest outline in tile widths
def inside_outside(model, names, archive):
    inside, outside, distances = [], [], []
    for name in names:
        with h5py.File(os.path.join(h5_dir, name)) as f:
            features = f["features"][0]
            # the tile size in level-0 px differs per file: 256 on 20x scans up to 1024 on 80x
            width = f["coords"].attrs["patch_size_level0"]
            centres = f["coords"][0] + width / 2
        polygons = outlines(archive, name[:-len(".h5")])
        tumour = np.zeros(len(features), bool)
        for polygon in polygons:
            tumour |= Path(polygon).contains_points(centres)
        with torch.no_grad():
            tile_logits = model(torch.from_numpy(features).to(device))[1].cpu().numpy()
        # edges reach 18,834 px on val, so the nearest outline point alone would overstate the distance: fill in points
        # a quarter tile apart along each edge, including the closing edge from the last point to the first
        points = [np.linspace(start, end, int(np.linalg.norm(end - start) // (width / 4)) + 2)
                  for polygon in polygons for start, end in zip(polygon, np.roll(polygon, -1, axis=0))]
        nearest = NearestNeighbors(n_neighbors=1).fit(np.concatenate(points))
        inside.append(tile_logits[tumour])
        outside.append(tile_logits[~tumour])
        distances.append(nearest.kneighbors(centres[~tumour])[0][:, 0] / width)
    return np.concatenate(inside), np.concatenate(outside), np.concatenate(distances)


def run_outline_test():
    model = TileClassifier().to(device)
    model.load_state_dict(torch.load(checkpoint))
    slides = pd.read_csv(splits_csv)
    rows = []
    with h5py.File(tumour_cache) as cache, zipfile.ZipFile(kather_zip) as archive:
        patients = slides.drop_duplicates("patient")
        patients = patients[(patients.split == "val") & patients.patient.isin(list(cache))]
        for patient, label, site in zip(patients.patient, patients.label, patients.site):
            inside, outside, distances = inside_outside(model, slides.h5[slides.patient == patient], archive)
            # a different count means this mask is not the one the model trained on
            if len(inside) != len(cache[patient]["features"]):
                raise ValueError(f"{patient}: {len(inside)} tiles inside, {len(cache[patient]['features'])} in the cache")
            row = {"label": label, "site": site, "tiles inside": len(inside), "tiles outside": len(outside),
                   "inside": inside.mean(), "outside": outside.mean()}
            for band, (near, far) in bands.items():
                part = outside[(distances >= near) & (distances < far)]
                row[band] = part.mean() if len(part) >= min_tiles else np.nan
            rows.append(row)
    rows = pd.DataFrame(rows)
    print(f"{len(rows)} val patients ({(rows.label == positive).sum()} ILC), {rows['tiles inside'].sum():,} tiles "
          f"inside, {rows['tiles outside'].sum():,} outside\n")
    print("median over patients of the mean tile logit")
    print(rows.groupby(["site", "label"])[["inside", "outside"]].median().round(3).to_string())
    aurocs = []
    for site, part in [("all", rows)] + list(rows.groupby("site")):
        y = part.label == positive
        aurocs.append({"site": site, "ILC": y.sum(), "NST": (~y).sum(), "inside": roc_auc_score(y, part.inside),
                       "outside": roc_auc_score(y, part.outside)})
    print("\nAUROC of the patient logit")
    print(pd.DataFrame(aurocs).round(3).to_string(index=False))
    # positive when the tiles inside point further towards the patient's own label
    gap = (rows.inside - rows.outside) * np.where(rows.label == positive, 1, -1)
    print(f"\ntowards the patient's label: stronger inside in {(gap > 0).mean():.3f} of patients, median gap "
          f"{gap.median():.3f}")
    table = []
    for band in bands:
        # patients with fewer than min_tiles tiles in the band have NaN there and are left out
        present = rows.dropna(subset=[band])
        for site, part in [("all", present)] + list(present.groupby("site")):
            y = (part.label == positive).to_numpy()
            table.append({"band": band, "site": site, "ILC": y.sum(), "NST": (~y).sum(),
                          "median ILC": part.loc[y, band].median(), "median NST": part.loc[~y, band].median(),
                          "AUROC": roc_auc_score(y, part[band]),
                          "AUROC inside, same patients": roc_auc_score(y, part.inside)})
    print(f"\nby distance from the nearest outline in tile widths, patients with at least {min_tiles} tiles in the band")
    print(pd.DataFrame(table).round(3).to_string(index=False))


load_or_run("outline_test", run_outline_test)

**Ranked tiles.** For ILC and for NST val patients: the most ILC-like, most NST-like and nearest-0 tiles. Ranking only picks tiles to look at; it does not show they are right. The tiles come from the GDC tile API (no login, no SVS download) into the git-ignored `data/external/gdc_tiles/`; the endpoint is undocumented, so it may change.

In [ ]:
# one patient can otherwise fill a list: TCGA-BH-A0HK holds 48% of the val tiles above logit 6
per_patient = 3
shown = 10
# the lower the key, the higher the tile in the list; one logit has one axis, so low on both classes means near 0
rankings = {"most ILC": lambda t: -t, "most NST": lambda t: t, "nearest 0": np.abs}


def patient_rows(group, tile_logits):
    # group: the patient's group in the tumour tile cache
    # tile_logits: logit of each tile of this patient, in the order of the cache
    slide_names = group.attrs["slides"]
    coords = group["coords"][()]
    slide = group["slide"][()]
    rows = []
    for ranking, key in rankings.items():
        keys = key(tile_logits)
        for index in np.argsort(keys)[:per_patient]:
            name = slide_names[slide[index]]
            rows.append({"ranking": ranking, "key": keys[index], "h5": name, "slide": name.split(".")[0],
                         "x": coords[index, 0], "y": coords[index, 1], "tile logit": tile_logits[index]})
    return rows


def top_tiles():
    model = TileClassifier().to(device)
    model.load_state_dict(torch.load(checkpoint))
    model.eval()
    slides = pd.read_csv(splits_csv)
    rows = []
    with h5py.File(tumour_cache) as cache:
        # patients without a group have no tumour tile
        patients = slides.drop_duplicates("patient")[["patient", "label", "split"]]
        patients = patients[(patients["split"] == "val") & patients["patient"].isin(list(cache))]
        for patient, label in zip(patients["patient"], patients["label"]):
            with torch.no_grad():
                patient_logit, tile_logits = model(torch.from_numpy(cache[patient]["features"][()]).to(device))
            for row in patient_rows(cache[patient], tile_logits.cpu().numpy()):
                rows.append({**row, "label": label, "patient logit": patient_logit.item()})
    rows = pd.DataFrame(rows)
    lists = {(ranking, label): rows[(rows["ranking"] == ranking) & (rows["label"] == label)].nsmallest(shown, "key")
             for ranking in rankings for label in ["ILC", "NST"]}
    return patients, lists


def run_rank_tiles():
    patients, lists = top_tiles()
    print(f"{len(patients)} val patients ({(patients['label'] == positive).sum()} ILC), at most {per_patient} tiles per "
          f"patient in each list, x and y in level-0 px")
    for (ranking, label), part in lists.items():
        print(f"\n{ranking} tiles, {label} patients")
        print(part[["slide", "x", "y", "tile logit", "patient logit"]].round(3).to_string(index=False))


load_or_run("rank_tiles", run_rank_tiles)

In [ ]:
gdc_tile_api = "https://api.gdc.cancer.gov/tile"
files_json = os.path.join(root, "data/meta/files_slides.json")
# GDC file id per .svs file name
with open(files_json) as f:
    file_ids = {hit["file_name"]: hit["file_id"] for hit in json.load(f)["data"]["hits"]}
out_dir = os.path.join(root, "data/external/gdc_tiles")
# width of GDC's tiles at full resolution, and of our tiles on 40x scans
grid = 512
columns = 5


@functools.cache
def full_level(file_id):
    # file_id: GDC file id of the slide
    with urllib.request.urlopen(f"{gdc_tile_api}/metadata/{file_id}", timeout=60) as response:
        metadata = json.load(response)
    # every crop of a fetched tile assumes this tile size and overlap
    if metadata["TileSize"] != str(grid) or metadata["Overlap"] != "1":
        raise ValueError(f"{file_id}: unexpected tile metadata {metadata}")
    # Deep Zoom doubles the size per level from 1 px at level 0, so full resolution is the first power of two at or above
    # the longer side
    return math.ceil(math.log2(max(int(metadata["Width"]), int(metadata["Height"]))))


def gdc_tile(file_id, column, row):
    # file_id: GDC file id of the slide
    # column, row: position on GDC's 512 px grid at full resolution
    # returns the path of the fetched .png, named by its level-0 corner; a tile on disk is not fetched again
    path = os.path.join(out_dir, f"{file_id}_{column * grid}_{row * grid}.png")
    if not os.path.exists(path):
        url = f"{gdc_tile_api}/{file_id}?level={full_level(file_id)}&x={column}&y={row}"
        with urllib.request.urlopen(url, timeout=60) as response:
            image = response.read()
        # written aside and renamed, so a thread fetching the same tile never reads a half-written file
        part = f"{path}.{threading.get_ident()}.part"
        with open(part, "wb") as f:
            f.write(image)
        os.replace(part, path)
    return path


def fetch_tile(name, x, y):
    # name: .h5 file name of the slide
    # x: left edge of the tile in level-0 px
    # y: top edge of the tile in level-0 px
    # only 40x tiles line up with GDC's grid; none of the six lists holds another scan type, so no cropping for them
    with h5py.File(os.path.join(h5_dir, name)) as f:
        tile_width = f["coords"].attrs["patch_size_level0"]
    if tile_width != grid or x % grid or y % grid:
        raise ValueError(f"{name}: tile at {x}, {y} of width {tile_width} does not line up with the {grid} px grid")
    return gdc_tile(file_ids[name[:-len(".h5")] + ".svs"], x // grid, y // grid)


def run_contact_sheets():
    _, lists = top_tiles()
    os.makedirs(out_dir, exist_ok=True)
    for (ranking, label), tiles in lists.items():
        rows = math.ceil(len(tiles) / columns)
        figure = Figure(figsize=(3 * columns, 3.3 * rows))
        axes = figure.subplots(rows, columns, squeeze=False)
        for axis in axes.flat:
            axis.axis("off")
        for axis, (_, tile) in zip(axes.flat, tiles.iterrows()):
            image = fetch_tile(tile["h5"], tile["x"], tile["y"])
            # a tile carries 1 px of each neighbour, so none on the left or top edge of the slide
            left, top = int(tile["x"] > 0), int(tile["y"] > 0)
            axis.imshow(imread(image)[top:top + grid, left:left + grid])
            axis.set_title(f"{tile['slide'][5:12]}: tile {tile['tile logit']:.2f}, patient {tile['patient logit']:.2f}",
                           fontsize=9)
        figure.suptitle(f"{ranking} tiles, {label} patients")
        path = os.path.join(out_dir, f"{ranking} tiles {label} patients".replace(" ", "_") + ".png")
        figure.savefig(path, dpi=170, bbox_inches="tight")
        print(path, flush=True)


# the sheets run_contact_sheets writes, one per list
sheets = [os.path.join(out_dir, f"{ranking} tiles {label} patients".replace(" ", "_") + ".png")
          for ranking in rankings for label in ["ILC", "NST"]]
load_or_run("contact_sheets", run_contact_sheets, sheets)
for sheet in sheets:
    show(sheet)

## 5. Slide heatmaps

Each figure: the slide, its tile logits, and the logits over the slide, with the Kather outlines drawn. The model trained only on tiles inside the outlines, so orange outside them is not ILC evidence, and the pattern inside a slide is not validated. White vertical strips over tissue are tiles missing from the `.h5` files, not low logits.

Triptychs: the first 10 test slides per class, as examples that inform no choice. Heatmaps: four val patients picked for what the review or the ranking shows.

In [ ]:
triptych_dir = os.path.join(root, "data/external/gdc_triptychs")
# per class, as the first 10 test slides are all NST
n_slides = 10
# longest thumbnail side in px; a panel is only about 890 px wide
thumbnail_side = 1024
# one colour scale for all slides, set on val: the tile logits' 5th and 95th percentiles are -6.25 and 6.04
logit_limit = 6
nst_ilc = LinearSegmentedColormap.from_list("nst_ilc", ["#2a78d6", "#f0efec", "#eb6834"])


def thumbnail(file_id):
    # file_id: GDC file id of the slide; returns the thumbnail and its level-0 px per thumbnail px
    path = os.path.join(triptych_dir, f"{file_id}.json")
    if not os.path.exists(path):
        with urllib.request.urlopen(f"{gdc_tile_api}/metadata/{file_id}", timeout=60) as response:
            content = response.read()
        with open(path, "wb") as f:
            f.write(content)
    with open(path) as f:
        metadata = json.load(f)
    # the crop below assumes this tile size and overlap
    if metadata["TileSize"] != str(grid) or metadata["Overlap"] != "1":
        raise ValueError(f"{file_id}: unexpected tile metadata {metadata}")
    width, height = int(metadata["Width"]), int(metadata["Height"])
    full_level = math.ceil(math.log2(max(width, height)))
    halvings = max(0, math.ceil(math.log2(max(width, height) / thumbnail_side)))
    path = os.path.join(triptych_dir, f"{file_id}.png")
    if not os.path.exists(path):
        image = np.ones((math.ceil(height / 2 ** halvings), math.ceil(width / 2 ** halvings), 3))
        for row in range(math.ceil(image.shape[0] / grid)):
            for column in range(math.ceil(image.shape[1] / grid)):
                url = f"{gdc_tile_api}/{file_id}?level={full_level - halvings}&x={column}&y={row}"
                with urllib.request.urlopen(url, timeout=60) as response:
                    tile = imread(BytesIO(response.read()), format="png")
                # tiles overlap their neighbours by 1 px, none at the left and top edge
                top, left = int(row > 0), int(column > 0)
                part = tile[top:top + grid, left:left + grid, :3]
                image[row * grid:row * grid + part.shape[0], column * grid:column * grid + part.shape[1]] = part
        imsave(path, image)
    image = imread(path)[:, :, :3]
    # from the saved file, which may have been made with another thumbnail_side
    return image, 2 ** round(math.log2(width / image.shape[1]))


def slide_logits(model, name, archive, device):
    # name: .h5 file name of the slide; archive: the opened Kather BRCA.zip
    with h5py.File(os.path.join(h5_dir, name)) as f:
        features, coords, tile_width = f["features"][0], f["coords"][0], f["coords"].attrs["patch_size_level0"]
    with torch.no_grad():
        tile_logits = model(torch.from_numpy(features).to(device))[1].cpu().numpy()
    return coords, tile_width, tile_logits, outlines(archive, name[:-len(".h5")])


def triptych(image, scale, coords, tile_width, tile_logits, polygons, title, path):
    # image, scale: from thumbnail; coords, tile_width, tile_logits, polygons: from slide_logits; path: .png to write
    corner = coords.min(0)
    if ((coords - corner) % tile_width).any():
        raise ValueError(f"tiles of width {tile_width} are not on a regular grid")
    column, row = ((coords - corner) // tile_width).T
    cells = np.full((row.max() + 1, column.max() + 1), np.nan)
    cells[row, column] = tile_logits
    # left, right, bottom, top in level-0 px, as imshow takes it
    left, top = corner
    cell_extent = (left, left + cells.shape[1] * tile_width, top + cells.shape[0] * tile_width, top)
    slide_extent = (0, image.shape[1] * scale, image.shape[0] * scale, 0)
    # at least 5 in high, so a wide slide's colour bar keeps its label
    figure = Figure(figsize=(15, max(5, 1 + 4.5 * image.shape[0] / image.shape[1])), layout="constrained")
    axes = figure.subplots(1, 3, sharex=True, sharey=True)
    norm = Normalize(-logit_limit, logit_limit)
    # NaN maps to fully transparent
    colours = nst_ilc(norm(cells))
    axes[0].imshow(image, extent=slide_extent)
    axes[1].imshow(colours, extent=cell_extent, interpolation="nearest")
    # weak logits are see-through
    colours[..., 3] = np.where(np.isnan(cells), 0, 0.8 * np.minimum(np.abs(cells) / logit_limit, 1))
    axes[2].imshow(image, extent=slide_extent)
    axes[2].imshow(colours, extent=cell_extent, interpolation="nearest")
    for axis in axes[1:]:
        for polygon in polygons:
            axis.plot(*np.vstack([polygon, polygon[:1]]).T, color="#0b0b0b", linewidth=0.8)
    for axis, heading in zip(axes, ["slide", "tile logit", "tile logit over the slide"]):
        axis.set(title=heading, xlim=slide_extent[:2], ylim=slide_extent[2:])
        axis.axis("off")
    figure.colorbar(ScalarMappable(norm, nst_ilc), ax=axes, shrink=0.7, extend="both",
                    ticks=[-logit_limit, -logit_limit / 2, 0, logit_limit / 2, logit_limit],
                    label="tile logit: NST (blue) to ILC (orange)")
    figure.suptitle(title)
    figure.savefig(path, dpi=200)


def run_triptychs():
    chosen = pd.read_csv(splits_csv).query("split == 'test'").groupby("label").head(n_slides)
    model = TileClassifier().to(device)
    model.load_state_dict(torch.load(checkpoint))
    model.eval()
    os.makedirs(triptych_dir, exist_ok=True)
    with h5py.File(tumour_cache) as cache, zipfile.ZipFile(kather_zip) as archive:
        for patient, label, name in zip(chosen["patient"], chosen["label"], chosen["h5"]):
            with torch.no_grad():
                patient_logit = model(torch.from_numpy(cache[patient]["features"][()]).to(device))[0].item()
            slide = name.split(".")[0]
            path = os.path.join(triptych_dir, f"{label}_{slide}.png")
            triptych(*thumbnail(file_ids[name[:-len(".h5")] + ".svs"]), *slide_logits(model, name, archive, device),
                     f"{slide}: labelled {label}, patient logit {patient_logit:.2f} (from tumour tiles)", path)
            print(path, flush=True)


# the figures run_triptychs writes, chosen as it chooses them
chosen = pd.read_csv(splits_csv).query("split == 'test'").groupby("label").head(n_slides)
triptychs = [os.path.join(triptych_dir, f"{label}_{name.split('.')[0]}.png")
             for label, name in zip(chosen["label"], chosen["h5"])]
load_or_run("triptychs", run_triptychs, triptychs)
for path in triptychs:
    show(path)

In [ ]:
heatmap_dir = os.path.join(root, "data/external/gdc_heatmaps")
# chosen for what the Thennavan review or the tile ranking shows, so not a sample of how the model does
reviews = {"TCGA-BH-A18L": "Mixed, no components listed", "TCGA-BH-A1FJ": "Mixed, IDC + ILC",
           "TCGA-BH-A0HK": "lobular", "TCGA-BH-A0B5": "lobular"}
note = "The model trained only on tiles inside the outline, so orange outside it is not ILC evidence."


def run_heatmaps():
    chosen = pd.read_csv(splits_csv).query("patient in @reviews")
    # test informs no choice
    if set(chosen["split"]) != {"val"} or chosen["patient"].nunique() != len(reviews):
        found = chosen[["patient", "split"]].to_dict("records")
        raise ValueError(f"expected only val slides of {list(reviews)}, got {found}")
    model = TileClassifier().to(device)
    model.load_state_dict(torch.load(checkpoint))
    model.eval()
    os.makedirs(triptych_dir, exist_ok=True)
    os.makedirs(heatmap_dir, exist_ok=True)
    with h5py.File(tumour_cache) as cache, zipfile.ZipFile(kather_zip) as archive:
        for patient, label, name in zip(chosen["patient"], chosen["label"], chosen["h5"]):
            with torch.no_grad():
                patient_logit = model(torch.from_numpy(cache[patient]["features"][()]).to(device))[0].item()
            slide = name.split(".")[0]
            path = os.path.join(heatmap_dir, f"{label}_{slide}.png")
            title = (f"{slide}: labelled {label}, review {reviews[patient]}, patient logit {patient_logit:.2f} "
                     f"(from tumour tiles)\n{note}")
            triptych(*thumbnail(file_ids[name[:-len(".h5")] + ".svs"]), *slide_logits(model, name, archive, device),
                     title, path)
            print(path, flush=True)


# the figures run_heatmaps writes
chosen = pd.read_csv(splits_csv).query("patient in @reviews")
heatmaps = [os.path.join(heatmap_dir, f"{label}_{name.split('.')[0]}.png")
            for label, name in zip(chosen["label"], chosen["h5"])]
load_or_run("heatmaps", run_heatmaps, heatmaps)
for path in heatmaps:
    show(path)

## 6. Site and pathologist review

**Site.** Train and val only, on patient mean features, as the patient logit is linear in them; saved to `data/uni2/patient_means.npz` (git-ignored, derived from UNI2-h). The features carry the site and the site alone weakly predicts the train label, but on val the model ranks as well within a site as across sites, and site explains no more of the logit than shuffled sites do. Val has only 2 sites, so this says nothing about stain styles elsewhere.

In [ ]:
means_path = os.path.join(root, "data/uni2/patient_means.npz")
# fewer patients than this leaves too few per fold or too noisy a site mean
min_site = 10
min_site_class = 5
folds = 5
# site shuffles in part 4
shuffles = 2000


def site_prediction(features, sites, title):
    # features: (n, 1536) patient means; sites: site of each patient; title: what to call this run in the printout
    counts = pd.Series(sites).value_counts()
    kept = counts[counts >= min_site]
    keep = pd.Series(sites).isin(kept.index).to_numpy()
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    cv = StratifiedKFold(folds, shuffle=True, random_state=seed)
    predicted = cross_val_predict(model, features[keep], sites[keep], cv=cv)
    print(f"{title}: {keep.sum()} patients, {len(kept)} sites; accuracy {(predicted == sites[keep]).mean():.3f} vs "
          f"{kept.iloc[0] / keep.sum():.3f} for always the largest site; balanced accuracy "
          f"{balanced_accuracy_score(sites[keep], predicted):.3f} vs {1 / len(kept):.3f} by chance")


def pair_aurocs(table):
    # table: one row per patient with its site, y (1 = ILC) and logit
    ilc, nst = table[table["y"] == 1], table[table["y"] == 0]
    # 1 where the ILC patient scores higher, 0.5 for a tie
    wins = (np.sign(ilc["logit"].to_numpy()[:, None] - nst["logit"].to_numpy()[None, :]) + 1) / 2
    same = ilc["site"].to_numpy()[:, None] == nst["site"].to_numpy()[None, :]
    return wins[same].mean(), wins[~same].mean(), same.sum(), (~same).sum()


def eta_squared(values, groups):
    # values: one number per patient; groups: group of each patient
    frame = pd.DataFrame({"value": values, "group": groups})
    means = frame.groupby("group")["value"].transform("mean")
    return ((means - frame["value"].mean()) ** 2).sum() / ((frame["value"] - frame["value"].mean()) ** 2).sum()


def run_site_confound():
    slides = pd.read_csv(splits_csv)
    patients = slides.drop_duplicates("patient")[["patient", "site", "label", "split"]]
    patients = patients[patients["split"].isin(["train", "val"])]
    if patients["site"].isna().any():
        raise ValueError("patient without a site")
    if not os.path.exists(means_path):
        with h5py.File(tumour_cache) as cache:
            patients = patients[patients["patient"].isin(list(cache))]
            means = np.stack([cache[patient]["features"][()].mean(0) for patient in patients["patient"]])
        np.savez(means_path, patient=patients["patient"].to_numpy().astype(str), means=means)
    saved = np.load(means_path)
    patients = patients.set_index("patient").loc[saved["patient"]].reset_index()
    means = saved["means"]
    if not np.isfinite(means).all():
        raise ValueError("non-finite patient mean")

    state = torch.load(checkpoint, map_location="cpu")
    patients["logit"] = means @ state["head.weight"].numpy()[0] + state["head.bias"].item()
    patients["y"] = (patients["label"] == positive).astype(int)
    is_train = (patients["split"] == "train").to_numpy()
    train, val = patients[is_train], patients[~is_train]
    print(f"{len(train)} train patients in {train['site'].nunique()} sites, {len(val)} val patients in "
          f"{val['site'].nunique()} sites; sites shared: {len(set(train['site']) & set(val['site']))}")

    print("\n1. the site from the features, train only, cross-validated")
    site_prediction(means[is_train], train["site"].to_numpy(), "all train patients")
    nst = is_train & (patients["y"] == 0).to_numpy()
    site_prediction(means[nst], patients["site"].to_numpy()[nst], "train NST only, so class cannot help")

    print("\n2. the label from the site alone, train only: the ILC share of the patient's site, without the patient")
    in_site = train.groupby("site")["y"].transform("count")
    ilc_in_site = train.groupby("site")["y"].transform("sum")
    several = in_site > 1
    share = (ilc_in_site - train["y"])[several] / (in_site[several] - 1)
    print(f"AUROC {roc_auc_score(train['y'][several], share):.3f} on {several.sum()} patients; {(~several).sum()} patients "
          f"are alone in their site and left out. On val this cannot be used: no site is shared.")

    print("\n3. val, ILC against NST: pairs from one site against pairs from two sites")
    within, across, n_within, n_across = pair_aurocs(val)
    print(f"all pairs {roc_auc_score(val['y'], val['logit']):.3f}; same site {within:.3f} ({n_within} pairs); different sites "
          f"{across:.3f} ({n_across} pairs)")

    print("\n4. val, share of the logit's variance")
    print(f"explained by class: {eta_squared(val['logit'], val['y']):.3f}")
    rng = np.random.default_rng(seed)
    for name, y in [("NST", 0), ("ILC", 1)]:
        part = val[val["y"] == y]
        counts = part["site"].value_counts()
        part = part[part["site"].isin(counts[counts >= min_site_class].index)]
        observed = eta_squared(part["logit"], part["site"])
        labels = part["site"].to_numpy()
        null = np.array([eta_squared(part["logit"], rng.permutation(labels)) for _ in range(shuffles)])
        print(f"within {name}, explained by site: {observed:.3f}, {len(part)} patients in {part['site'].nunique()} sites with "
              f"at least {min_site_class}; shuffled sites give {np.median(null):.3f} ({np.percentile(null, 2.5):.3f}-"
              f"{np.percentile(null, 97.5):.3f}), and {(null >= observed).mean():.4f} of them reach it")


load_or_run("site_confound", run_site_confound)

**Pathologist review.** Evaluation only: the val patient logits against the Thennavan review, which covers all 171 val patients. On patients reviewed as purely lobular or ductal the model ranks about as well against the review as against the TCGA label. Errors are reviewed as Mixed far more often than correct patients are. The spread of tile logits inside a patient does not tell Mixed from pure.

In [ ]:
folder = os.path.join(root, "data/external/thennavan2021")
lobular, ductal, mixed = "Invasive lobular carcinoma", "Invasive ductal carcinoma", "Mixed"


def run_thennavan():
    features = pd.read_excel(f"{folder}/mmc3.xlsx", dtype=str, keep_default_na=False)
    features["patient"] = features["Sample CLID"].str.strip().str[:12]
    if features["patient"].duplicated().any():
        raise ValueError("a patient with several reviewed samples")
    features["review"] = features["2016 Histology Annotations"].str.strip()
    for column in ["Epithelial tubule formation", "Nuclear pleomorphism", "Mitosis"]:
        features[column + " score"] = pd.to_numeric(features[column].str.extract(r"score = (\d)")[0])
    features["components"] = features["Components if mixed"].str.strip()
    keep = ["patient", "review", "components", "Epithelial tubule formation score", "Nuclear pleomorphism score",
            "Mitosis score"]

    state = torch.load(checkpoint, map_location="cpu")
    w = state["head.weight"].numpy()[0]
    b = state["head.bias"].item()
    slides = pd.read_csv(splits_csv)
    patients = slides.drop_duplicates("patient")[["patient", "label", "split"]]
    patients = patients[patients["split"] == "val"]
    rows = []
    with h5py.File(tumour_cache) as cache:
        patients = patients[patients["patient"].isin(list(cache))]
        for patient in patients["patient"]:
            t = cache[patient]["features"][()] @ w + b
            rows.append({"patient": patient, "logit": t.mean(), "tile std": t.std(), "share positive": (t > 0).mean()})
    val = patients.merge(pd.DataFrame(rows), on="patient").merge(features[keep], on="patient", how="left")
    val["review"] = val["review"].fillna("not reviewed")
    ilc = val["label"] == positive
    print(f"{len(val)} val patients, AUROC against our label {roc_auc_score(ilc, val['logit']):.3f}")

    print("\nour label vs review")
    print(pd.crosstab(val["review"], val["label"]).to_string())

    print("\nall val errors against our label")
    errors = val[(ilc & (val["logit"] <= 0)) | (~ilc & (val["logit"] > 0))].sort_values("logit", ascending=False)
    print(errors[["patient", "label", "review", "components", "logit"]].round(3).to_string(index=False))
    print("\nreview of val patients the model got right, for comparison")
    right = val.drop(errors.index)
    print(pd.crosstab(right["review"], right["label"]).to_string())

    pure = val[val["review"].isin([lobular, ductal])]
    y_review = (pure["review"] == lobular).to_numpy()
    y_ours = (pure["label"] == positive).to_numpy()
    disagree = pure[y_review != y_ours]
    print(f"\nreviewed as purely lobular or ductal: {len(pure)} ({y_review.sum()} lobular); ours and the review disagree on "
          f"{len(disagree)}")
    print(disagree[["patient", "label", "review", "logit"]].round(3).to_string(index=False))
    for name, y in [("review", y_review), ("our label", y_ours)]:
        print(f"AUROC on these patients against {name}: {roc_auc_score(y, pure['logit']):.3f}")

    both = val[val["review"].isin([lobular, ductal, mixed])]
    y_mixed = (both["review"] == mixed).to_numpy()
    print(f"\nAUROC of tile std for Mixed vs pure: {roc_auc_score(y_mixed, both['tile std']):.3f}, "
          f"{y_mixed.sum()} Mixed, {(~y_mixed).sum()} pure")
    print("\nreviewed Mixed val patients")
    print(val[val["review"] == mixed][["patient", "label", "components", "logit", "tile std", "share positive"]]
          .sort_values("logit").round(3).to_string(index=False))

    part = val[val["review"] == ductal]
    print(f"\nreviewed ductal (n={len(part)}), Spearman of patient logit with the review's grade scores")
    for column in ["Epithelial tubule formation score", "Nuclear pleomorphism score", "Mitosis score"]:
        present = part[[column, "logit"]].dropna()
        print(f"  {column}: {present[column].corr(present['logit'], method='spearman'):.3f} (n={len(present)})")


load_or_run("thennavan", run_thennavan)

## 7. Sparse autoencoder concepts ⏱

A top-k sparse autoencoder on the stored UNI2-h embeddings: 1536 → 4096 concepts, of which the 32 largest per tile are kept; the decoder columns have unit norm. It passes the embedding through unchanged (`x_recon + (x - x_recon).detach()`), so the head scores as without it, while LRP reaches the embedding only through the concepts. Saved to `models/sae_k32_capped.pt`; it derives from UNI2-h, so it falls under that licence (git-ignored, never redistributed).

- Data: from each train patient, up to 1,000 tiles inside the outlines and 1,000 outside for ILC and 250 each for NST, so the classes come out about balanced (at most about 3 GB in RAM); from each val patient, up to 500 inside and 500 outside. Seed 0.
- Trained on the MSE (AdamW, learning rate 1e-3, batches of 4,096) for at most 30 epochs, stopping 2 epochs after the best val MSE and keeping that epoch. It then prints, for val tiles inside and outside the outlines, the MSE and median cosine, and raises if a kept concept is negative.

Then each tile logit is split into concepts, `t = b + Σ z_k (w·d_k) + w·e`, with `d_k` the decoder column of concept k and `e` the reconstruction error; the split raises if it is off by more than 0.01. Over all val tumour tiles the concepts are ranked by their share of |contributions|; the top 20, each with its side (the sign of `w·d_k`) and the 3 tiles it fires on most, are saved to `data/external/gdc_tiles/sae_top_tiles.csv` and go on one contact sheet, framed orange for the ILC side and blue for NST.

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, device):
        # device: where the SAE and every input goes
        super().__init__()
        self.device = device
        self.encoder = nn.Linear(1536, 4096)
        self.decoder = nn.Linear(4096, 1536, bias=False)
        self.normalize_decoder()
        self.to(device)

    def normalize_decoder(self):
        with torch.no_grad():
            self.decoder.weight /= self.decoder.weight.norm(dim=0, keepdim=True)

    def forward(self, x):
        z = self.encoder(x)
        # keeps the 32 largest concepts of each tile and zeroes the rest
        z_bar = z * torch.zeros_like(z).scatter(-1, torch.topk(z, 32, dim=-1).indices, 1.0)
        x_recon = self.decoder(z_bar)
        # x_out equals x, so the head scores as without the SAE, while LRP reaches x only through the concepts
        x_out = x_recon + (x - x_recon).detach()
        return x_out, x_recon, z_bar

    def fit(self, x, val):
        # x: (n, 1536) float32 train embeddings
        # val: (m, 1536) float32 val embeddings; stops 2 epochs after their best MSE and keeps that epoch's weights
        x, val = torch.from_numpy(x), torch.from_numpy(val)
        optimiser = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=0.0)
        best_loss, best_state, best_epoch = np.inf, None, None
        for epoch in range(30):
            order = torch.randperm(len(x))
            total = 0.0
            fired = torch.zeros(4096, dtype=torch.bool, device=self.device)
            for i in range(0, len(order), 4096):
                batch = x[order[i:i + 4096]].to(self.device)
                _, x_recon, z_bar = self(batch)
                loss = nn.functional.mse_loss(x_recon, batch)
                if not torch.isfinite(loss):
                    raise ValueError(f"loss is {loss.item()} in epoch {epoch + 1}")
                optimiser.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
                optimiser.step()
                self.normalize_decoder()
                total += loss.item() * len(batch)
                fired |= (z_bar != 0).any(0)
            val_total = 0.0
            with torch.inference_mode():
                for i in range(0, len(val), 4096):
                    batch = val[i:i + 4096].to(self.device)
                    val_total += nn.functional.mse_loss(self(batch)[1], batch).item() * len(batch)
            val_loss = val_total / len(val)
            if not np.isfinite(val_loss):
                raise ValueError(f"non-finite val loss in epoch {epoch + 1}")
            print(f"epoch {epoch + 1}/30: train mse {total / len(order):.5f}, val mse {val_loss:.5f}, "
                  f"dead concepts {(~fired).float().mean().item():.1%}", flush=True)
            if val_loss < best_loss:
                best_loss, best_epoch = val_loss, epoch
                best_state = {name: value.clone() for name, value in self.state_dict().items()}
            elif epoch - best_epoch >= 2:
                break
        self.load_state_dict(best_state)
        print(f"best epoch {best_epoch + 1}: val mse {best_loss:.5f}", flush=True)

In [ ]:
sae_path = os.path.join(root, "models/sae_k32_capped.pt")
# tiles per patient, from inside and again from outside the outlines; about class-balanced on train
train_caps = {"ILC": 1000, "NST": 250}
val_caps = {"ILC": 500, "NST": 500}


def sample_tiles(slides, caps, archive, rng):
    # slides: splits.csv rows of one split
    # caps: most tiles per patient by label, for inside and for outside the outlines
    # archive: the opened Kather BRCA.zip
    # rng: picks the tiles
    inside, outside = [], []
    for _, rows in slides.groupby("patient"):
        features, tumour = [], []
        for name in rows["h5"]:
            with h5py.File(os.path.join(h5_dir, name)) as f:
                features.append(f["features"][0])
                # the tile size in level-0 px differs per file: 256 on 20x scans up to 1024 on 80x
                centres = f["coords"][0] + f["coords"].attrs["patch_size_level0"] / 2
            if not np.isfinite(features[-1]).all():
                raise ValueError(f"non-finite features in {name}")
            tumour.append(np.zeros(len(centres), bool))
            for polygon in outlines(archive, name[:-len(".h5")]):
                tumour[-1] |= Path(polygon).contains_points(centres)
        features, tumour = np.concatenate(features), np.concatenate(tumour)
        cap = caps[rows["label"].iloc[0]]
        for picked, part in [(inside, features[tumour]), (outside, features[~tumour])]:
            picked.append(part[rng.choice(len(part), min(cap, len(part)), replace=False)])
    return np.concatenate(inside), np.concatenate(outside)


def measure(sae, x, label):
    # sae: the trained TopKSAE
    # x: (n, 1536) embeddings
    # label: name of the printed row
    recon = []
    with torch.inference_mode():
        for i in range(0, len(x), 4096):
            _, x_recon, z_bar = sae(torch.from_numpy(x[i:i + 4096]).to(sae.device))
            # the encoder has no ReLU; a negative kept activation would flip its concept, and the rankings read larger
            # as stronger
            if (z_bar < 0).any():
                raise ValueError(f"{label}: a kept concept has a negative activation")
            recon.append(x_recon.cpu().numpy())
    recon = np.concatenate(recon)
    cosine = (x * recon).sum(1) / (np.linalg.norm(x, axis=1) * np.linalg.norm(recon, axis=1))
    print(f"{label}: {len(x)} tiles, mse {((x - recon) ** 2).mean(dtype=np.float64):.5f}, "
          f"median cosine {np.median(cosine):.3f}", flush=True)


def run_train_sae():
    slides = pd.read_csv(splits_csv)
    with h5py.File(tumour_cache) as cache:
        slides = slides[slides["patient"].isin(list(cache))]
    rng = np.random.default_rng(seed)
    with zipfile.ZipFile(kather_zip) as archive:
        train = np.concatenate(sample_tiles(slides[slides["split"] == "train"], train_caps, archive, rng))
        val_inside, val_outside = sample_tiles(slides[slides["split"] == "val"], val_caps, archive, rng)
    torch.manual_seed(seed)
    sae = TopKSAE(device)
    sae.fit(train, np.concatenate((val_inside, val_outside)))
    sae.eval()
    # saved before measuring, so a later failure cannot lose the weights
    torch.save({name: value.cpu() for name, value in sae.state_dict().items()}, sae_path)
    measure(sae, val_inside, "val inside")
    measure(sae, val_outside, "val outside")


load_or_run("train_sae", run_train_sae, [sae_path])

In [ ]:
top_tiles_csv = os.path.join(out_dir, "sae_top_tiles.csv")


def top_concepts(sae, w, b, patients):
    # sae: the trained TopKSAE
    # w, b: weights and bias of the tile classifier's head
    # patients: val patients, one row each with patient and label
    w_t = torch.from_numpy(w).float().to(sae.device)
    # how much one unit of each concept moves the logit
    v = sae.decoder.weight.detach().T @ w_t
    size = torch.zeros(4096, device=sae.device)
    fired = torch.zeros(4096, device=sae.device)
    best_z = torch.full((3, 4096), -np.inf, device=sae.device)
    best_id = torch.zeros((3, 4096), dtype=torch.long, device=sae.device)
    starts, seen, gap = [], 0, 0.0
    with h5py.File(tumour_cache) as cache, torch.inference_mode():
        for patient in patients["patient"]:
            _, x_recon, z = sae(torch.from_numpy(cache[patient]["features"][()]).to(sae.device))
            # the split t = b + Σ z_k v_k + w·(x - x_recon) holds only if the reconstruction's logit is Σ z_k v_k
            gap = max(gap, (x_recon @ w_t - z @ v).abs().max().item())
            size += (z * v).abs().sum(0)
            fired += (z != 0).sum(0)
            top_z, top_row = z.topk(min(3, len(z)), dim=0)
            best_z, order = torch.cat((best_z, top_z)).topk(3, dim=0)
            best_id = torch.cat((best_id, top_row + seen)).gather(0, order)
            starts.append(seen)
            seen += len(z)
    if gap > 1e-2:
        raise ValueError(f"the logit split is off by {gap}")
    share, fires = (size / size.sum()).cpu().numpy(), (fired / seen).cpu().numpy()
    v, best_z, best_id = v.cpu().numpy(), best_z.cpu().numpy(), best_id.cpu().numpy()
    print(f"top 20 concepts by share of |contributions| over {seen} val tumour tiles, each with its 3 strongest tiles:")
    rows = []
    with h5py.File(tumour_cache) as cache:
        for rank, k in enumerate(np.argsort(-share)[:20]):
            side = "ILC" if v[k] > 0 else "NST"
            print(f"{rank + 1}. concept {k}: share {share[k]:.3f}, fires on {fires[k]:.3f} of tiles, v {v[k]:.3f} "
                  f"({side} side)")
            for z, tile in zip(best_z[:, k], best_id[:, k]):
                index = np.searchsorted(starts, tile, side="right") - 1
                patient, label = patients.iloc[index]
                group, row = cache[patient], tile - starts[index]
                x, y = group["coords"][row]
                slide = group.attrs["slides"][group["slide"][row]]
                logit = group["features"][row] @ w + b
                print(f"   {slide[:-len('.h5')]} ({label}) x {x} y {y}: activation {z:.2f}, tile logit {logit:.2f}")
                rows.append({"rank": rank + 1, "concept": k, "share": share[k], "side": side, "slide": slide,
                             "label": label, "x": x, "y": y, "activation": z, "tile logit": logit})
    return pd.DataFrame(rows)


def run_sae_concepts():
    slides = pd.read_csv(splits_csv)
    with h5py.File(tumour_cache) as cache:
        slides = slides[slides["patient"].isin(list(cache))]
    sae = TopKSAE(device)
    sae.load_state_dict(torch.load(sae_path, map_location=device))
    sae.eval()
    state = torch.load(checkpoint, map_location="cpu")
    w, b = state["head.weight"][0].numpy().astype(np.float64), state["head.bias"][0].item()
    tiles = top_concepts(sae, w, b, slides[slides["split"] == "val"].drop_duplicates("patient")[["patient", "label"]])
    tiles.to_csv(top_tiles_csv, index=False)


load_or_run("sae_concepts", run_sae_concepts, [top_tiles_csv])

In [ ]:
sheet_path = os.path.join(out_dir, "sae_top_concepts.png")
# the diverging pair validated for the heatmaps; red is avoided as it blends into pink H&E
side_colours = {"ILC": "#eb6834", "NST": "#2a78d6"}


def concept_sheet(tiles, images, path, notes=None):
    # tiles: rows from top_concepts, 3 per concept in rank order
    # images: path of each tile's fetched image, in the order of tiles
    # path: .png file to write
    # notes: optional text per concept id, added under its row label
    figure = Figure(figsize=(9, 3.2 * len(tiles) // 3))
    axes = figure.subplots(len(tiles) // 3, 3, squeeze=False)
    for axis, (_, tile), image in zip(axes.flat, tiles.iterrows(), images):
        # a tile carries 1 px of each neighbour, so none on the left or top edge of the slide
        left, top = int(tile["x"] > 0), int(tile["y"] > 0)
        axis.imshow(imread(image)[top:top + grid, left:left + grid])
        axis.set_xticks([])
        axis.set_yticks([])
        # the frame marks the side the concept pushes the logit to; the row label says it in words
        for spine in axis.spines.values():
            spine.set_edgecolor(side_colours[tile["side"]])
            spine.set_linewidth(3)
        axis.set_title(f"{tile['slide'][5:12]} ({tile['label']}): activation {tile['activation']:.1f}, "
                       f"tile {tile['tile logit']:.2f}", fontsize=8)
    for row, (_, tile) in zip(axes, tiles.iloc[::3].iterrows()):
        label = f"#{tile['rank']} concept {tile['concept']}\n{tile['side']} side, share {tile['share']:.1%}"
        if notes is not None:
            label += "\n" + notes[tile["concept"]]
        row[0].set_ylabel(label, fontsize=9)
    figure.savefig(path, dpi=120, bbox_inches="tight")


def run_concept_sheet():
    tiles = pd.read_csv(top_tiles_csv)
    images = [fetch_tile(tile["slide"], tile["x"], tile["y"]) for _, tile in tiles.iterrows()]
    concept_sheet(tiles, images, sheet_path)


load_or_run("concept_sheet", run_concept_sheet, [sheet_path])
show(sheet_path)

## 8. CONCH labels for the SAE concepts

Labels each of the top 20 concepts zero-shot with CONCH (gated on Hugging Face, weights never redistributed), from the average image embedding of its 3 tiles. Terms compete within four groups (tissue, architecture, nuclei, cytoplasm); each term is the average text embedding of a few descriptions filled into CONCH's templates, as in CONCH's prompt ensembles. A concept takes a group's best term, or `unresolved` when the runner-up is within 0.02. Writes `conch_concept_labels.csv` and the concept sheet with the labels under each row. Reads no test slide.

Known limits, each with a way to improve it:

- Labels are relative: each term's mean over the 20 concepts is subtracted, because some terms win on almost any tile. Neutral reference concepts would make them absolute.
- 3 tiles per concept, often from one slide; concept #2 has a blank background tile among its 3. More tiles, with background filtered, would steady the image embedding.
- `min_margin` 0.02 is a guess, to be checked on high-, medium- and low-margin tiles.
- The vocabulary is hand-written; it could follow the WHO criteria and be checked against the Thennavan scores (evaluation only). The nuclei terms showed no ILC/NST signal.
- A 128 µm tile is too small for targetoid growth.
- Positive control: CONCH's own "invasive lobular carcinoma" vs "invasive ductal carcinoma" prompts separate ILC from NST tiles at AUROC 0.79 on all tiles and 0.85 on tumour tiles, so the tiles and CONCH are fine and the vocabulary is the weak part.

In [ ]:
conch_csv = os.path.join(out_dir, "conch_concept_labels.csv")
conch_sheet_path = os.path.join(out_dir, "sae_top_concepts_conch.png")
# from CONCH's prompts/crc100k_prompts_all_per_class.json
templates = ["{}.", "a photomicrograph of {}.", "an H&E image of {}.", "{}, H&E."]
# a first draft, American spelling as in the PubMed captions CONCH was trained on. Architecture, nuclei and cytoplasm
# only mean something where tissue says tumor; targetoid growth needs larger crops than one 128 um tile
groups = {
    "tissue": {
        "tumor": ["tumor-rich tissue", "invasive breast carcinoma"],
        "collagenous stroma": ["collagenous stroma", "dense fibrous stroma"],
        "lymphocytes": ["lymphocyte-rich tissue", "a dense lymphocytic infiltrate"],
        "adipose tissue": ["adipose tissue", "fat cells"],
        "benign epithelium": ["benign breast epithelium", "normal breast ducts and lobules"],
    },
    "architecture": {
        "single-file rows": ["tumor cells arranged in single-file rows",
                             "single-cell-wide linear rows of tumor cells infiltrating breast stroma"],
        "scattered single cells": ["scattered individual tumor cells in stroma",
                                   "isolated single tumor cells infiltrating stroma"],
        "cohesive nests": ["cohesive nests of tumor cells", "tumor cells in cohesive clusters"],
        "glands and tubules": ["tumor cells forming glands", "tubule formation by tumor cells"],
        "solid sheets": ["solid sheets of tumor cells", "tumor cells growing in solid sheets"],
    },
    "nuclei": {
        "uniform round": ["tumor cells with small uniform round nuclei",
                          "monotonous tumor cells with bland round nuclei"],
        "pleomorphic": ["tumor cells with enlarged pleomorphic nuclei",
                        "marked nuclear pleomorphism with prominent nucleoli"],
    },
    "cytoplasm": {
        "signet ring": ["signet ring cells", "tumor cells with intracytoplasmic vacuoles pushing the nucleus aside"],
        "eosinophilic granular": ["tumor cells with abundant eosinophilic granular cytoplasm", "apocrine tumor cells"],
        "vacuolated foamy": ["tumor cells with foamy vacuolated cytoplasm", "histiocytoid tumor cells"],
    },
}
min_margin = 0.02


def embed_tiles(model, preprocess, images):
    # model: CONCH model in eval mode
    # preprocess: CONCH's image transform, PIL image to tensor
    # images: tiles as RGB PIL images
    # returns (n_images, d) unit-length embeddings in CONCH's joint image-text space
    device = next(model.parameters()).device
    parts = []
    with torch.inference_mode():
        for i in range(0, len(images), 32):
            batch = torch.stack([preprocess(image) for image in images[i:i + 32]]).to(device)
            parts.append(model.encode_image(batch, proj_contrast=True, normalize=True).cpu())
    return torch.cat(parts).numpy()


def embed_terms(model, terms):
    # model: CONCH model in eval mode
    # terms: term name to its descriptions, each filled into every template
    # returns (n_terms, d) unit-length text embeddings, one per term
    prompts = [template.format(text) for texts in terms.values() for text in texts for template in templates]
    # conch's tokenize() calls batch_encode_plus, which transformers 5 removed; this is its body, calling the tokenizer
    tokenizer = get_tokenizer()
    tokens = tokenizer(prompts, max_length=127, add_special_tokens=True, return_token_type_ids=False, truncation=True,
                       padding="max_length", return_tensors="pt")["input_ids"]
    # CONCH's context is 128 tokens with the last reserved for its <cls>, so a <pad> holds that place
    tokens = torch.nn.functional.pad(tokens, (0, 1), value=tokenizer.pad_token_id)
    with torch.inference_mode():
        tokens = tokens.to(next(model.parameters()).device)
        embeddings = model.encode_text(tokens, normalize=True).cpu()
    # as in CONCH's ensembles: average each term's unit-length prompt embeddings, then renormalise
    counts = [len(texts) * len(templates) for texts in terms.values()]
    means = torch.stack([part.mean(0) for part in embeddings.split(counts)])
    return torch.nn.functional.normalize(means, dim=-1).numpy()


def run_conch_labels():
    tiles = pd.read_csv(top_tiles_csv)
    paths = [fetch_tile(tile["slide"], tile["x"], tile["y"]) for _, tile in tiles.iterrows()]
    images = []
    for (_, tile), path in zip(tiles.iterrows(), paths):
        # a tile carries 1 px of each neighbour, so none on the left or top edge of the slide
        left, top = int(tile["x"] > 0), int(tile["y"] > 0)
        with Image.open(path) as image:
            rgb = image.convert("RGB")
        if rgb.width < left + grid or rgb.height < top + grid:
            raise ValueError(f"{tile['slide']} at {tile['x']}, {tile['y']}: {rgb.size} is smaller than a full tile")
        images.append(rgb.crop((left, top, left + grid, top + grid)))
    model, preprocess = create_model_from_pretrained("conch_ViT-B-16", "hf_hub:MahmoodLab/conch", device=device)
    model.eval()
    # an SAE concept's embedding is the average of its tiles' unit-length embeddings, renormalised
    by_concept = pd.DataFrame(embed_tiles(model, preprocess, images)).groupby(tiles["concept"].to_numpy(), sort=False)
    concepts = torch.nn.functional.normalize(torch.from_numpy(by_concept.mean().to_numpy()), dim=-1).numpy()
    table = tiles.drop_duplicates("concept")[["rank", "concept", "side", "share"]].reset_index(drop=True)
    for group, terms in groups.items():
        # both sides are unit length, so these are cosine similarities
        scores = concepts @ embed_terms(model, terms).T
        if not np.isfinite(scores).all():
            raise ValueError(f"non-finite CONCH scores in {group}")
        # some terms win on almost any tile; taking out each term's mean over the concepts leaves what sets one apart
        scores = scores - scores.mean(0)
        ranked = np.sort(scores, axis=1)
        margin = ranked[:, -1] - ranked[:, -2]
        table[group] = np.where(margin >= min_margin, np.array(list(terms))[scores.argmax(1)], "unresolved")
        table[f"{group} margin"] = margin
        for name, column in zip(terms, scores.T):
            table[f"{group}: {name}"] = column
    table.to_csv(conch_csv, index=False)
    print(table[["rank", "concept", "side", *groups]].to_string(index=False))
    notes = {row["concept"]: f"{row['tissue']}, {row['architecture']}\n{row['nuclei']}, {row['cytoplasm']}"
             for _, row in table.iterrows()}
    concept_sheet(tiles, paths, conch_sheet_path, notes)


load_or_run("conch_labels", run_conch_labels, [conch_csv, conch_sheet_path])
show(conch_sheet_path)

## 9. LRP from SAE concepts to pixels ⏱

CP-LRP from one SAE concept through UNI2-h to the pixels, with zennit plus rules adapted from lxt (AttnLRP): the identity rule on SiLU and on LayerNorm's detached std, detached q and k in attention so v carries all relevance, the uniform rule on SwiGLU's gated product; Gamma on UNI2-h's layers and Epsilon on the SAE and the head. Relevance flows back only through the chosen concept; all others are masked. Needs the UNI2-h weights from `download.py`.

`run_lrp_tiles` maps the 3 concepts that carry each tile's logit most, on every val tile in `data/external/gdc_tiles/`. It checks that the patches leave the forward unchanged, that relevance is conserved, and that the concepts on one tile give different maps, and saves the maps to `lrp_maps.npz` (git-ignored, derived from UNI2-h). Opens no test tile. The tiles it maps are those on disk, so a fresh run maps fewer: some came from scripts not in this notebook.

The figure shows, per tile, the tile as UNI2-h sees it and the maps of its 3 strongest concepts, 13 tiles per page. Orange pushes the tile logit towards ILC, blue towards NST. Each map has its own symmetric scale, so compare where, not how much, across maps.

`patch_for_lrp` changes timm's and zennit's classes for the whole kernel, and step 10 relies on it; restart the kernel before rerunning anything above with the original forward.

In [ ]:
encoder_weights = os.path.join(root, "models/uni2-h/pytorch_model.bin")
# kept when this cell is rerun: the patches stay in the kernel until it restarts
_patched = globals().get("_patched", False)


# _IdentityRule and _DivideGradient follow lxt/efficient/rules.py (lxt 2.1, Clear BSD, Achtibat et al., AttnLRP,
# ICML 2024); lxt itself is not imported, as importing lxt.efficient pulls in transformers
class _IdentityRule(Function):
    # identity rule on an element-wise non-linearity: relevance passes through unchanged
    @staticmethod
    def forward(ctx, fn, x):
        out = fn(x)
        ctx.save_for_backward(out / (x + 1e-10))
        return out

    @staticmethod
    def backward(ctx, grad):
        return None, ctx.saved_tensors[0] * grad


class _DivideGradient(Function):
    # uniform rule: each of the `factor` operands of a product gets 1/factor of the relevance
    @staticmethod
    def forward(ctx, x, factor):
        ctx.factor = factor
        return x

    @staticmethod
    def backward(ctx, grad):
        return grad / ctx.factor, None


def load_encoder(device):
    # device: where UNI2-h runs
    encoder = timm.create_model(pretrained=False, model_name="vit_giant_patch14_224", img_size=224, patch_size=14,
                                depth=24, num_heads=24, init_values=1e-5, embed_dim=1536, mlp_ratio=2.66667 * 2,
                                num_classes=0, no_embed_class=True, mlp_layer=timm.layers.SwiGLUPacked,
                                act_layer=nn.SiLU, reg_tokens=8, dynamic_img_size=True)
    encoder.load_state_dict(torch.load(encoder_weights, map_location="cpu"), strict=True)
    return encoder.eval().requires_grad_(False).to(device)


def patch_for_lrp():
    # swaps the forward of timm's Attention, GluMlp and LayerNorm and of nn.SiLU, plus zennit's BasicHook, for CP-LRP in
    # the gradient*input framework; process-global, and a second call would wrap the forwards twice
    global _patched
    if _patched:
        raise RuntimeError("patch_for_lrp already ran in this process")
    _patched = True
    # nn.SiLU is UNI2-h's SwiGLU activation: identity rule
    silu_forward = nn.SiLU.forward
    nn.SiLU.forward = lambda self, x: _IdentityRule.apply(lambda t: silu_forward(self, t), x)

    # a detached std is the identity rule on the normalization
    def layer_norm_forward(self, x):
        std = (x.var(dim=-1, keepdim=True, unbiased=False) + self.eps).sqrt()
        x = (x - x.mean(dim=-1, keepdim=True)) / std.detach()
        if self.weight is not None:
            x = x * self.weight
        if self.bias is not None:
            x = x + self.bias
        return x
    for cls in (nn.LayerNorm, TimmLayerNorm):
        cls.forward = layer_norm_forward

    # manual matmul path, as fused SDPA is opaque to the hooks; detached q and k leave v to carry all relevance (CP-LRP)
    def attention_forward(self, x, attn_mask=None, is_causal=False):
        if self.gate is not None or attn_mask is not None or is_causal:
            raise NotImplementedError("the LRP attention patch covers UNI2-h only: no gate, mask or causal attention")
        B, N, _ = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4).unbind(0)
        q, k = self.q_norm(q.detach()), self.k_norm(k.detach())
        attn = self.attn_drop(((q * self.scale) @ k.transpose(-2, -1)).softmax(dim=-1))
        x = (attn @ v).transpose(1, 2).reshape(B, N, self.attn_dim)
        return self.proj_drop(self.proj(self.norm(x)))
    TimmAttention.forward = attention_forward

    # uniform rule on the gated product of SwiGLU
    def glu_mlp_forward(self, x):
        x1, x2 = self.fc1(x).chunk(2, dim=self.chunk_dim)
        x = _DivideGradient.apply(x1 * self.act(x2) if self.gate_last else self.act(x1) * x2, 2)
        return self.drop2(self.fc2(self.norm(self.drop1(x))))
    GluMlp.forward = glu_mlp_forward

    # zennit's rules pass relevance where the patches above pass gradients (relevance = gradient * input): convert the
    # incoming gradient to relevance (times output) and the outgoing relevance back to a gradient (divided by input)
    original_forward, original_backward = BasicHook.forward, BasicHook.backward

    def hook_forward(self, module, args, kwargs, output):  # zennit reads these 4 names to register with kwargs
        original_forward(self, module, args, kwargs, output)
        self.stored_tensors["output"] = output.detach()

    def hook_backward(self, module, grad_input, grad_output):
        relevance = original_backward(self, module, grad_input, (grad_output[0] * self.stored_tensors["output"],))
        x = self.stored_tensors["input"][0].detach()
        return tuple(None if r is None else r / stabilize(x, 1e-10) for r in relevance)
    BasicHook.forward, BasicHook.backward = hook_forward, hook_backward


def lrp_composite():
    # xai_cc's A_gam_c0_l025 on UNI2-h (t14.py:117): Gamma 0 on the patch embedding, 0.25 on its Linear layers; Epsilon on
    # the SAE and the head
    def module_map(ctx, name, module):
        if name.startswith("encoder.") and isinstance(module, nn.Conv2d):
            return Gamma(0.0)
        if name.startswith("encoder.") and isinstance(module, nn.Linear):
            return Gamma(0.25)
        if isinstance(module, nn.Linear):
            return Epsilon(1e-6)
        return None
    return Composite(module_map=module_map)


def concept_heatmap(parts, image, concept, composite):
    # parts: nn.ModuleDict of "encoder", "sae" and "head", after patch_for_lrp
    # image: (1, 3, 224, 224) tile, normalised
    # concept: the one SAE concept relevance may flow back through; all others are masked
    # composite: from lrp_composite
    x = image.detach().requires_grad_(True)
    with composite.context(parts):
        x_out, _, z_bar = parts["sae"](parts["encoder"](x))
        # zero the gradient of every concept but this one, as xai_cc's _make_mask_hook (evaluation/sae/plotting.py:402-406)
        z_bar.register_hook(lambda g: g * (torch.arange(g.shape[-1], device=g.device) == concept))
        t = parts["head"](x_out)[:, 0]
        # a gradient of 1 makes the output relevance the logit itself (gradient*input)
        t.backward(torch.ones_like(t))
    # per pixel, summed over the colour channels
    relevance = (x * x.grad).sum(dim=1)[0]
    if not torch.isfinite(relevance).all():
        raise ValueError(f"concept {concept}: relevance is not finite for {(~torch.isfinite(relevance)).sum().item()} pixels")
    return relevance.detach().cpu().numpy()

In [ ]:
tile_dir = os.path.join(root, "data/external/gdc_tiles")
maps_path = os.path.join(tile_dir, "lrp_maps.npz")
# the preprocessing that reproduced the stored vectors: median cosine 1.0000, median logit gap 0.0064 on 60 val tiles
to_tensor = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])


def val_tiles():
    with open(files_json) as f:
        slide_names = {hit["file_id"]: hit["file_name"] for hit in json.load(f)["data"]["hits"]}
    splits = pd.read_csv(splits_csv).set_index("h5")["split"]
    names, stored = [], []
    for name in sorted(os.listdir(tile_dir)):
        # the contact sheets in the same folder have no file id in their name
        file_id, *position = name[:-len(".png")].split("_")
        h5_name = slide_names.get(file_id, "").removesuffix(".svs") + ".h5"
        if splits.get(h5_name) != "val":
            continue
        with h5py.File(os.path.join(h5_dir, h5_name)) as f:
            rows = np.flatnonzero((f["coords"][0] == np.array(position, dtype=int)).all(1))
            # a tile inside the strips has no stored vector, so the model never scored it
            if len(rows):
                names.append(name)
                stored.append(f["features"][0, rows[0]])
    return names, np.stack(stored)


def load_image(name):
    # name: file name <file id>_<x>_<y>.png
    _, x, y = name[:-len(".png")].split("_")
    with Image.open(os.path.join(tile_dir, name)) as image:
        rgb = np.asarray(image.convert("RGB"))
    # a tile carries 1 px of each neighbour, so none on the left or top edge of the slide
    left, top = int(int(x) > 0), int(int(y) > 0)
    crop = Image.fromarray(rgb[top:top + 512, left:left + 512])
    return to_tensor(crop.resize((256, 256), Image.Resampling.BICUBIC).resize((224, 224), Image.Resampling.BILINEAR))


def embed(encoder, images):
    # encoder: UNI2-h
    # images: (n, 3, 224, 224) tiles, normalised
    with torch.inference_mode():
        return torch.cat([encoder(images[i:i + 16]) for i in range(0, len(images), 16)])


def run_lrp_tiles():
    names, stored = val_tiles()
    images = torch.stack([load_image(name) for name in names]).to(device)
    encoder = load_encoder(device)
    sae = TopKSAE(device)
    sae.load_state_dict(torch.load(sae_path, map_location=device))
    sae.eval().requires_grad_(False)
    classifier = TileClassifier()
    classifier.load_state_dict(torch.load(checkpoint, map_location="cpu"))
    head = classifier.head.eval().requires_grad_(False).to(device)

    before = embed(encoder, images)
    patch_for_lrp()
    after = embed(encoder, images)
    print(f"{len(names)} val tiles; patched forward, largest change in the embedding "
          f"{(after - before).abs().max().item():.1e}, in the logit {(head(after) - head(before)).abs().max().item():.1e}",
          flush=True)
    cosines = torch.cosine_similarity(after, torch.from_numpy(stored).to(device))
    print(f"cosine with the stored vector: median {cosines.median().item():.4f}, lowest {cosines.min().item():.4f}",
          flush=True)

    with torch.inference_mode():
        z_bar = sae(after)[2]
    # how much one unit of each concept moves the logit
    v = sae.decoder.weight.T @ head.weight[0]
    contributions = z_bar * v
    top = contributions.abs().topk(3, dim=1).indices
    parts = nn.ModuleDict({"encoder": encoder, "sae": sae, "head": head})
    composite = lrp_composite()
    maps = []
    for i, name in enumerate(names):
        maps.append([concept_heatmap(parts, images[[i]], k.item(), composite) for k in top[i]])
        print(f"tile {i + 1}/{len(names)}: {name}", flush=True)
    maps = np.stack(maps)

    chosen = contributions.gather(1, top).cpu().numpy()
    bias = sae.encoder.bias[top].cpu().numpy() * v[top].cpu().numpy()
    totals = maps.sum((2, 3))
    print(f"relevance sum over contribution z_k v_k: median {np.median(totals / chosen):.3f}, percentiles 10/90 "
          f"{np.percentile(totals / chosen, 10):.3f} {np.percentile(totals / chosen, 90):.3f}", flush=True)
    print(f"relevance sum over contribution without the encoder bias, v_k (z_k - b_k): median "
          f"{np.median(totals / (chosen - bias)):.3f}, percentiles 10/90 {np.percentile(totals / (chosen - bias), 10):.3f} "
          f"{np.percentile(totals / (chosen - bias), 90):.3f}", flush=True)

    # per 14 px patch, the scale UNI2-h sees; absolute, since a concept's sign says nothing about where it looks
    patches = maps.reshape(len(names), 3, 16, 14, 16, 14).sum((3, 5))
    correlations = [pd.DataFrame(patch.reshape(3, -1).T).corr(method="spearman").abs().to_numpy()[np.triu_indices(3, 1)]
                    for patch in patches]
    print(f"absolute rank correlation between concepts on one tile, median of tile medians "
          f"{np.median(np.median(correlations, axis=1)):.3f}", flush=True)

    share = np.abs(chosen).sum(1) / contributions.abs().sum(1).cpu().numpy()
    print(f"the 3 concepts' share of |contributions| per tile: median {np.median(share):.3f}", flush=True)
    np.savez(maps_path, names=np.array(names), concepts=top.cpu().numpy(), contributions=chosen, maps=maps)


load_or_run("lrp_tiles", run_lrp_tiles, [maps_path])

In [ ]:
# the diverging pair validated for the heatmaps, with a neutral midpoint; red is avoided as it blends into pink H&E
colours = LinearSegmentedColormap.from_list("nst_ilc", ["#2a78d6", "#f0efec", "#eb6834"])
rows_per_page = 13


def lrp_tile_image(name):
    # name: file name <file id>_<x>_<y>.png; cropped and resized as for UNI2-h (load_image)
    _, x, y = name[:-len(".png")].split("_")
    with Image.open(os.path.join(tile_dir, name)) as image:
        rgb = np.asarray(image.convert("RGB"))
    left, top = int(int(x) > 0), int(int(y) > 0)
    crop = Image.fromarray(rgb[top:top + 512, left:left + 512])
    return crop.resize((256, 256), Image.Resampling.BICUBIC).resize((224, 224), Image.Resampling.BILINEAR)


def run_lrp_figure():
    data = np.load(maps_path)
    names, concepts, contributions, maps = data["names"], data["concepts"], data["contributions"], data["maps"]
    with open(files_json) as f:
        slide_names = {hit["file_id"]: hit["file_name"] for hit in json.load(f)["data"]["hits"]}
    for page, start in enumerate(range(0, len(names), rows_per_page)):
        chunk = range(start, min(start + rows_per_page, len(names)))
        figure = Figure(figsize=(10, 2.6 * len(chunk)), layout="constrained")
        axes = figure.subplots(len(chunk), 4, squeeze=False)
        for row, i in zip(axes, chunk):
            for axis in row:
                axis.set_xticks([])
                axis.set_yticks([])
            file_id, x, y = names[i][:-len(".png")].split("_")
            row[0].imshow(lrp_tile_image(names[i]))
            row[0].set_ylabel(f"{slide_names[file_id][5:12]}\n{x}, {y}", fontsize=8)
            for axis, k, contribution, relevance in zip(row[1:], concepts[i], contributions[i], maps[i]):
                limit = np.percentile(np.abs(relevance), 99.5)
                axis.imshow(relevance, cmap=colours, vmin=-limit, vmax=limit)
                axis.set_title(f"concept {k}: z v {contribution:.2f}", fontsize=8)
        figure.suptitle("orange: towards ILC, blue: towards NST; each map scaled to its own 99.5th percentile of "
                        "|relevance|", fontsize=9)
        path = os.path.join(tile_dir, f"lrp_maps_{page + 1}.png")
        figure.savefig(path, dpi=110, bbox_inches="tight")
        print(path, flush=True)


# the pages run_lrp_figure writes, one per rows_per_page tiles
pages = [os.path.join(tile_dir, f"lrp_maps_{page + 1}.png")
         for page in range(math.ceil(len(np.load(maps_path)["names"]) / rows_per_page))]
load_or_run("lrp_figure", run_lrp_figure, pages)
for page in pages:
    show(page)

## 10. PCX prototypes on the SAE concepts ⏱

PCX (Dreyer et al.): per class, a Gaussian mixture of 4 prototypes over the concept contributions `z_k v_k` of train tumour tiles (up to 500 per patient). On this model they are what LRP, gradient × input and Integrated Gradients (baseline 0) all give at the concept layer, as everything after it is linear. Two variants on the same tiles: `raw` contributions, and `l1` with each tile's vector divided by its summed |contributions|. Each class is explained by its own evidence: the logit for ILC, minus the logit for NST.

For each variant `run_pcx_tiles` prints each prototype's share, concept logit and leading concepts, checks that the prototypes come back on two disjoint halves of the train patients, and scores val tiles. It picks from every train tumour tile each prototype's 8 shown tiles, by Euclidean distance and by density, and 6 reference tiles per concept (RelMax). Saves to `models/pcx_prototypes_<variant>.npz`, the report to `pcx_report_<variant>.txt` and the picked tiles to `pcx_tiles_<variant>.csv` (all git-ignored, derived from UNI2-h).

`run_pcx_figure` draws the concept × prototype matrix as xai_cc does: top row each prototype's shown tiles, left column each concept's reference tiles cropped to the largest blob of the concept's LRP map, cells the prototype mean of the concept (% of a tile's |contributions| for `l1`, logits for `raw`). One PDF per variant, tile pick, class and row order: 16 in `data/external/gdc_tiles/`. Jupyter cannot show PDFs inline without another dependency, so the cell lists their paths.

In [ ]:
pcx_path = os.path.join(root, "models/pcx_prototypes_{}.npz")
report_path = os.path.join(root, "data/external/gdc_tiles/pcx_report_{}.txt")
tiles_path = os.path.join(root, "data/external/gdc_tiles/pcx_tiles_{}.csv")
tiles_per_patient = 500
prototypes = 4
# shown tiles per prototype and reference tiles per concept, as xai_cc (pcx_sae_cluster.py:188, pcx_sae_render.py:237)
nearest = 8
references = 6


def contributions(sae, v, patients, rng=None):
    # sae: the trained TopKSAE
    # v: (4096,) logit change per unit of each concept
    # patients: rows with patient
    # rng: picks up to tiles_per_patient tumour tiles per patient; None takes every tile
    # returns the 32 active concepts of each tile and their contributions, kept sparse, and (patient, row) per tile
    concepts, values, index = [], [], []
    with h5py.File(tumour_cache) as cache, torch.inference_mode():
        for patient in patients["patient"]:
            features = cache[patient]["features"]
            if rng is None:
                rows = range(len(features))
                # in slices, as one patient can have tens of thousands of tiles
                parts = (features[start:start + 16384] for start in range(0, len(features), 16384))
            else:
                # h5py reads only increasing indices
                rows = np.sort(rng.choice(len(features), min(tiles_per_patient, len(features)), replace=False))
                parts = [features[rows]]
            for part in parts:
                z = sae(torch.from_numpy(part).to(sae.device))[2]
                top = z.topk(32, dim=1).indices
                # concept ids fit in int16, which keeps 2.9 million tiles small
                concepts.append(top.cpu().numpy().astype(np.int16))
                values.append((z * v).gather(1, top).cpu().numpy())
            index += [(patient, row) for row in rows]
    return np.concatenate(concepts), np.concatenate(values), index


def densify(concepts, values, alive):
    # concepts, values: from contributions
    # alive: concepts that fire on the train tiles, the columns of the result
    column = np.full(4096, -1)
    column[alive] = np.arange(len(alive))
    dense = np.zeros((len(concepts), len(alive)), np.float32)
    rows, slots = np.nonzero(column[concepts] >= 0)
    dense[rows, column[concepts[rows, slots]]] = values[rows, slots]
    return dense


def normalise(x, variant):
    # x: (n, d) dense contributions
    # variant: "raw" keeps them; "l1" divides each tile by its summed |contributions|, so prototypes compare patterns,
    #   not size, as xai_cc's pcx_sae_cluster.py:135-140 (PCX App. C.5 may do the same, not verified)
    if variant == "raw":
        return x
    size = np.abs(x).sum(1, keepdims=True)
    if (size == 0).any():
        raise ValueError(f"{(size == 0).sum()} tiles have no contribution on the concepts that fire on the fit tiles")
    return x / size


def matched_cosines(a, b):
    # a, b: (prototypes, d) means of two fits; the pairing with the highest summed cosine
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    cosines = a @ b.T
    best = max(itertools.permutations(range(prototypes)), key=lambda p: cosines[range(prototypes), p].sum())
    return cosines[range(prototypes), best]


def distances(mixture, concepts, values, alive, variant):
    # mixture: a fitted diagonal GaussianMixture
    # concepts, values: sparse contributions of the tiles to score, as the class's evidence
    # alive, variant: as in densify and normalise
    # returns per tile and prototype the Euclidean distance to the mean, as the PCX tutorial and xai_cc
    # (pcx_sae_cluster.py:173), and the variance-weighted one; within a prototype the smallest variance-weighted distance
    # is the highest density, as the rest of its log density is the same for every tile
    euclidean, weighted = [], []
    for start in range(0, len(concepts), 32768):
        x = normalise(densify(concepts[start:start + 32768], values[start:start + 32768], alive), variant)
        euclidean.append(cdist(x, mixture.means_))
        weighted.append(np.stack([(((x - mean) ** 2) / var).sum(1)
                                  for mean, var in zip(mixture.means_, mixture.covariances_)], axis=1))
    return np.concatenate(euclidean), np.concatenate(weighted)


def pcx_top_concepts(raw, assigned, signed):
    # raw: (n, d) the fit tiles' contributions as the class's evidence, not normalised; assigned: prototype per tile
    # signed: False ranks concepts by |contribution| ("abs"), True by contribution ("signed")
    # returns the columns of raw that are among some prototype's 3 concepts with the largest mean over its 10 largest
    # member values, as xai_cc's pcx_sae_cluster.py:194-205
    picked = set()
    for c in range(prototypes):
        members = raw[assigned == c] if signed else np.abs(raw[assigned == c])
        k = min(10, len(members))
        largest = np.partition(members, len(members) - k, axis=0)[-k:].mean(0)
        picked |= set(np.argsort(-largest)[:3].tolist())
    return sorted(picked)


def ordered(means, rows, signed):
    # means: (prototypes, d) the mixture means; rows: columns of means from pcx_top_concepts; signed: as there
    # returns the concept x prototype matrix, its rows and its prototype order, sorted as xai_cc's
    # pcx_sae_cluster.py:207-224: prototypes by summed |value|, concepts by summed |value| or summed value
    matrix = means[:, rows].T
    columns = np.argsort(-np.abs(matrix).sum(0), kind="stable")
    matrix = matrix[:, columns]
    order = np.argsort(-(matrix.sum(1) if signed else np.abs(matrix).sum(1)), kind="stable")
    return matrix[order], np.array(rows)[order], columns


def locate(index, picks):
    # index: (patient, row) per tile; picks: positions in index
    # returns (slide, x, y) of each pick: its .h5 file name and top left corner in level-0 px
    with h5py.File(tumour_cache) as cache:
        return [(cache[p].attrs["slides"][cache[p]["slide"][r]], *cache[p]["coords"][r])
                for p, r in (index[i] for i in picks)]


def run_pcx_tiles():
    slides = pd.read_csv(splits_csv)
    with h5py.File(tumour_cache) as cache:
        slides = slides[slides["patient"].isin(list(cache))]
    patients = slides.drop_duplicates("patient")[["patient", "label", "split"]]
    sae = TopKSAE(device)
    sae.load_state_dict(torch.load(sae_path, map_location=device))
    sae.eval().requires_grad_(False)
    state = torch.load(checkpoint, map_location="cpu")
    w, b = state["head.weight"][0], state["head.bias"][0].item()
    v = sae.decoder.weight.T @ w.to(device)

    rng = np.random.default_rng(seed)
    train, val = patients[patients["split"] == "train"], patients[patients["split"] == "val"]
    train_concepts, train_values, train_index = contributions(sae, v, train, rng)
    val_concepts, val_values, val_index = contributions(sae, v, val, rng)
    alive = np.unique(train_concepts)
    label_of = dict(zip(patients["patient"], patients["label"]))
    train_labels = np.array([label_of[patient] for patient, _ in train_index])
    val_labels = np.array([label_of[patient] for patient, _ in val_index])
    # every train tumour tile, to pick the shown tiles and the reference tiles from
    all_concepts, all_values, all_index = contributions(sae, v, train)
    all_labels = np.array([label_of[patient] for patient, _ in all_index])
    size = np.abs(all_values).sum(1)
    if (size == 0).any():
        raise ValueError(f"{(size == 0).sum()} train tumour tiles have no contribution at all")
    print(f"{len(all_index)} train tumour tiles; share of their |contributions| on concepts that never fire on the fit "
          f"tiles, so outside the prototypes: {np.abs(all_values[~np.isin(all_concepts, alive)]).sum() / size.sum():.5f}",
          flush=True)
    saved = {}
    for variant in ["raw", "l1"]:
        lines = []

        def say(line):
            print(line, flush=True)
            lines.append(line)

        say(f"variant {variant}: train tiles {len(train_index)}, val tiles {len(val_index)}, concepts firing on train "
            f"{len(alive)}" + ("; means are shares of a tile's summed |contributions|" if variant == "l1" else "")
            + "; NST values are evidence for NST, minus the contributions")
        # means, variances and weights rebuild a class's mixture without a refit: GaussianMixture(covariance_type="diag")
        # with means_, covariances_ and weights_ set, and precisions_cholesky_ = variances ** -0.5
        saved[variant] = {"alive": alive, "classes": [], "weights": [], "means": [], "variances": [], "logits": []}
        # the logit is ILC's evidence, so NST's is its negative
        for label, sign in [("ILC", 1), ("NST", -1)]:
            raw = sign * densify(train_concepts[train_labels == label], train_values[train_labels == label], alive)
            x = normalise(raw, variant)
            index = [i for i, l in zip(train_index, train_labels) if l == label]
            mixture = GaussianMixture(prototypes, covariance_type="diag", random_state=seed).fit(x)
            assigned = mixture.predict(x)

            # stability: the same fit on two disjoint halves of this class's train patients
            names = train[train["label"] == label]["patient"].to_numpy()
            half = set(np.random.default_rng(seed).permutation(names)[:len(names) // 2])
            in_half = np.array([patient in half for patient, _ in index])
            halves = [GaussianMixture(prototypes, covariance_type="diag", random_state=seed).fit(x[part]).means_
                      for part in (in_half, ~in_half)]
            cosines = matched_cosines(*halves)

            val_x = normalise(sign * densify(val_concepts[val_labels == label], val_values[val_labels == label], alive),
                              variant)
            val_share = np.bincount(mixture.predict(val_x), minlength=prototypes) / len(val_x)
            say(f"\n{label}: {len(x)} train tiles; matched cosine between the two halves "
                f"{np.round(cosines, 3).tolist()}, median {np.median(cosines):.3f}; mean log-likelihood per tile "
                f"train {mixture.score(x):.1f}, val {mixture.score(val_x):.1f}")
            # the shown tiles, from every train tumour tile of the class: nearest by each distance
            in_class = np.flatnonzero(all_labels == label)
            for select, distance in zip(["euclidean", "density"], distances(
                    mixture, all_concepts[in_class], sign * all_values[in_class], alive, variant)):
                tiles = [locate(all_index, in_class[np.argsort(distance[:, c])[:nearest]]) for c in range(prototypes)]
                saved[variant][f"{label}_{select}_slides"] = np.array([[t[0] for t in ts] for ts in tiles])
                saved[variant][f"{label}_{select}_xy"] = np.array([[t[1:] for t in ts] for ts in tiles])
            for mode in ["abs", "signed"]:
                matrix, rows, columns = ordered(mixture.means_, pcx_top_concepts(raw, assigned, mode == "signed"),
                                                mode == "signed")
                saved[variant][f"{label}_{mode}_matrix"] = matrix
                saved[variant][f"{label}_{mode}_concepts"] = alive[rows]
                saved[variant][f"{label}_{mode}_columns"] = columns
            for c in range(prototypes):
                mean = mixture.means_[c]
                top = np.argsort(-np.abs(mean))[:5]
                # in logit units for both variants: the concept part of the logit of this prototype's tiles
                logit = np.median(b + sign * raw[assigned == c].sum(1))
                say(f"  prototype {c + 1}: share train {mixture.weights_[c]:.3f}, val {val_share[c]:.3f}; median "
                    f"concept logit of its tiles {logit:.2f}; leading concepts "
                    + ", ".join(f"{alive[k]} ({mean[k]:+.3f})" for k in top))
                saved[variant]["classes"].append(label)
                saved[variant]["weights"].append(mixture.weights_[c])
                saved[variant]["means"].append(mean)
                saved[variant]["variances"].append(mixture.covariances_[c])
                saved[variant]["logits"].append(logit)
        with open(report_path.format(variant), "w") as f:
            f.write("\n".join(lines) + "\n")

    # reference tiles, RelMax as PCX (CRP's reference_sampling with abs_norm) and xai_cc (pcx_sae_render.py:292-296):
    # for each concept in any figure, the train tumour tiles where it has the largest share of the tile's |contributions|.
    # Concepts never go negative, so each one's contributions share a sign: for concepts pushing toward a class this is
    # CRP's ranking, and the absolute share also ranks the others by how strongly they fire, not by how little
    shown = np.unique(np.concatenate([arrays[f"{label}_{mode}_concepts"] for arrays in saved.values()
                                      for label in ["ILC", "NST"] for mode in ["abs", "signed"]]))
    share = np.abs(all_values) / size[:, None]
    picked = []
    for k in shown:
        rows, slots = np.nonzero(all_concepts == k)
        picked.append(locate(all_index, rows[np.argsort(-share[rows, slots])[:references]]))
    for variant, arrays in saved.items():
        np.savez(pcx_path.format(variant), ref_concepts=shown,
                 ref_slides=np.array([[t[0] for t in ts] for ts in picked]),
                 ref_xy=np.array([[t[1:] for t in ts] for ts in picked]), **{k: np.array(a) for k, a in arrays.items()})
        # every picked tile in one table, as xai_cc's prototype_tiles.csv (pcx_sae_render.py:398-408); a TCGA slide name
        # starts with its patient's 12 character barcode
        table = [(label, select, f"prototype {c + 1}", rank + 1, str(slide)[:12], str(slide), x, y)
                 for label in ["ILC", "NST"] for select in ["euclidean", "density"]
                 for c, (slides, xy) in enumerate(zip(arrays[f"{label}_{select}_slides"], arrays[f"{label}_{select}_xy"]))
                 for rank, (slide, (x, y)) in enumerate(zip(slides, xy))]
        table += [("both", "reference", f"concept {k}", rank + 1, str(slide)[:12], str(slide), x, y)
                  for k, tiles in zip(shown, picked) for rank, (slide, x, y) in enumerate(tiles)]
        pd.DataFrame(table, columns=["class", "pick", "for", "rank", "patient", "slide", "x", "y"]).to_csv(
            tiles_path.format(variant), index=False)


load_or_run("pcx_tiles", run_pcx_tiles, [path.format(variant) for path in [pcx_path, report_path, tiles_path]
                                     for variant in ["raw", "l1"]])
# the per-variant reports, also when the step is not rerun
for variant in ["raw", "l1"]:
    with open(report_path.format(variant)) as f:
        print(f"\n{report_path.format(variant)}:\n{f.read()}")

In [ ]:
gdc_data_api = "https://api.gdc.cancer.gov/data"
svs_dir = os.path.join(root, "data/external/svs")
# 60x scans about 249,000 px tall: the tile API answers HTTP 500 for their metadata and tiles alike (2026-11-08), so
# tile_image reads them from the downloaded SVS
gdc_unservable = {"TCGA-OL-A66J-01Z-00-DX1.661F7F70-E4D4-4875-B8C4-556F7927F3BA.h5",
                  "TCGA-OL-A66K-01Z-00-DX1.C1DC85F1-4FAE-4411-9886-11DCB5E70CC3.h5"}
matrix_path = os.path.join(out_dir, "pcx_matrix_{}_{}_{}_{}.pdf")


def slide_path(name):
    # name: .h5 file name of a slide in gdc_unservable
    # returns the path of its .svs, downloaded from GDC once and checked against GDC's md5
    with open(files_json) as f:
        hit = next(h for h in json.load(f)["data"]["hits"] if h["file_name"] == name[:-len(".h5")] + ".svs")
    path = os.path.join(svs_dir, hit["file_name"])
    if not os.path.exists(path):
        os.makedirs(svs_dir, exist_ok=True)
        part, md5 = f"{path}.{threading.get_ident()}.part", hashlib.md5()
        with urllib.request.urlopen(f"{gdc_data_api}/{hit['file_id']}", timeout=60) as response, open(part, "wb") as f:
            while chunk := response.read(1 << 20):
                md5.update(chunk)
                f.write(chunk)
        if md5.hexdigest() != hit["md5sum"]:
            raise ValueError(f"{hit['file_name']}: md5 {md5.hexdigest()} is not GDC's {hit['md5sum']}")
        os.replace(part, path)
    return path


def tile_image(name, x, y):
    # name: .h5 file name of the slide
    # x, y: top left corner of the tile in level-0 px
    # returns the tile at full resolution as an RGB array, cut from the GDC tiles it overlaps, so any scan type works
    with h5py.File(os.path.join(h5_dir, name)) as f:
        width = int(f["coords"].attrs["patch_size_level0"])
    if name in gdc_unservable:
        with openslide.OpenSlide(slide_path(name)) as slide:
            return np.asarray(slide.read_region((x, y), 0, (width, width)).convert("RGB"))
    file_id = file_ids[name[:-len(".h5")] + ".svs"]
    columns = range(x // grid, (x + width - 1) // grid + 1)
    rows = range(y // grid, (y + width - 1) // grid + 1)
    canvas = np.zeros((len(rows) * grid, len(columns) * grid, 3), np.uint8)
    filled = np.zeros(canvas.shape[:2], bool)
    for i, row in enumerate(rows):
        for j, column in enumerate(columns):
            with Image.open(gdc_tile(file_id, column, row)) as image:
                rgb = np.asarray(image.convert("RGB"))
            # a tile carries 1 px of each neighbour, so none on the left or top edge of the slide
            piece = rgb[int(row > 0):int(row > 0) + grid, int(column > 0):int(column > 0) + grid]
            canvas[i * grid:i * grid + piece.shape[0], j * grid:j * grid + piece.shape[1]] = piece
            filled[i * grid:i * grid + piece.shape[0], j * grid:j * grid + piece.shape[1]] = True
    top, left = y - rows[0] * grid, x - columns[0] * grid
    if not filled[top:top + width, left:left + width].all():
        raise ValueError(f"{name}: tile at {x}, {y} of width {width} reaches past the edge of the slide")
    return canvas[top:top + width, left:left + width]


def model_input(rgb):
    # rgb: a tile at full resolution, from tile_image
    # returns the 224 px image UNI2-h sees and its normalised tensor; this reproduced the stored vectors of 256, 512 and
    # 768 px tiles (cosine 0.9999-1.0000 on 3 train tiles each)
    image = Image.fromarray(rgb).resize((256, 256), Image.Resampling.BICUBIC)
    image = image.resize((224, 224), Image.Resampling.BILINEAR)
    return image, to_tensor(image)


def stored_vector(slide, x, y):
    # slide: .h5 file name; x, y: the tile's top left corner in level-0 px
    with h5py.File(os.path.join(h5_dir, slide)) as f:
        rows = np.flatnonzero((f["coords"][0] == (x, y)).all(1))
        if len(rows) != 1:
            raise ValueError(f"{slide}: {len(rows)} stored tiles at {x}, {y}")
        return np.asarray(f["features"][0, rows[0]], np.float32)


# copied from xai_cc/src/xai_cc/pcx_sae_render.py:188-209 (the "median" reference crop); changed: takes the 224 px image
# and the heatmap rather than computing the heatmap itself
def reference_heatmap_crop(image, heatmap, sigma=3):
    # image: the 224 px PIL tile UNI2-h saw
    # heatmap: (224, 224) relevance of one concept on it
    # sigma: gaussian smoothing width applied to the heatmap before blob detection
    # smooth, threshold at the median positive value, keep the largest connected blob
    smoothed = gaussian_filter(heatmap, sigma=sigma)
    pos_vals = smoothed[smoothed > 0]
    thresh = np.median(pos_vals) if pos_vals.size else 0.0
    component_map, n_components = scipy_label(smoothed > thresh)
    if n_components > 0:
        pixel_counts = np.bincount(component_map.flatten())
        pixel_counts[0] = 0  # ignore background
        largest = pixel_counts.argmax()
        rows, cols = np.where(component_map == largest)
        r0, r1, c0, c1 = rows.min(), rows.max(), cols.min(), cols.max()
    else:
        r0, c0 = 0, 0
        r1, c1 = heatmap.shape[0] - 1, heatmap.shape[1] - 1
    # crop the tile to that blob's bounding box
    return image.crop((c0, r0, c1 + 1, r1 + 1))


# copied from xai_cc/src/xai_cc/pcx_sae_render.py:44-116; changed: the shown tiles come as images, not dataset rows; the
# matrix is numpy; raw cells print logits, not %; a Figure, not pyplot; the prototype labels and the path are passed
def plot_concept_prototype_matrix(prototype_images, ref_imgs, concept_matrix, top_concepts, result_path,
                                  prototype_labels, percent):
    n_concepts = len(top_concepts)
    n_prototypes = len(prototype_images)
    n_ref = len(next(iter(ref_imgs.values())))
    resize = torchvision.transforms.Resize((224, 224))  # native tile res; 120 softened the tiles

    fig = Figure(figsize=(n_prototypes + 6, n_concepts + 6), dpi=150)
    axs = fig.subplots(nrows=n_concepts + 1, ncols=n_prototypes + 1,
                       gridspec_kw={"width_ratios": [6] + [1] * n_prototypes, "height_ratios": [6] + [1] * n_concepts})

    # top row: prototype sample images, the raw tiles imshown directly
    axs[0, 0].axis("off")
    for p in range(n_prototypes):
        grid = make_grid([resize(torch.from_numpy(np.array(im)).permute(2, 0, 1)) for im in prototype_images[p]],
                         nrow=1, padding=0)
        axs[0, p + 1].imshow(grid.permute(1, 2, 0).cpu().numpy())
        axs[0, p + 1].set_anchor("N")  # top-align so short (small-cluster) columns fill from the top
        axs[0, p + 1].set_title(f"proto {prototype_labels[p]}")
        axs[0, p + 1].set(xticks=[], yticks=[])

    # left column: concept reference images (crops from reference_heatmap_crop), imshown directly
    for i, c in enumerate(top_concepts):
        grid = make_grid([resize(torch.from_numpy(np.array(im)).permute(2, 0, 1)) for im in ref_imgs[c]],
                         nrow=n_ref, padding=0)
        axs[i + 1, 0].imshow(grid.permute(1, 2, 0).cpu().numpy())
        axs[i + 1, 0].set_ylabel(f"concept {c}")
        axs[i + 1, 0].set(xticks=[], yticks=[])

    # inner cells: concept_matrix heatmap values
    max_abs = float(np.abs(concept_matrix).max())
    for i in range(n_concepts):
        for j in range(n_prototypes):
            val = concept_matrix[i, j]
            axs[i + 1, j + 1].matshow([[abs(val)]], vmin=0, vmax=max_abs, cmap="Reds" if val >= 0 else "Blues")
            axs[i + 1, j + 1].text(0, 0, f"{val * 100:.1f}" if percent else f"{val:.2f}", ha="center", va="center",
                                   color="white" if abs(val) > max_abs / 2 else "black", fontsize=15)
            axs[i + 1, j + 1].axis("off")

    fig.tight_layout()
    # dpi high so the embedded prototype/concept tile crops stay sharp
    fig.savefig(result_path, dpi=900, bbox_inches="tight")
    print(f"concept–prototype matrix saved to {result_path}", flush=True)


def run_pcx_figure():
    # read whole, so a pcx_tiles run can rewrite the files while this one runs
    data = {variant: dict(np.load(pcx_path.format(variant))) for variant in ["raw", "l1"]}

    # every tile first, so a failed GDC fetch stops the run before any LRP; a rerun skips the tiles on disk
    wanted = set()
    for d in data.values():
        for key in [f"{label}_{select}" for label in ["ILC", "NST"] for select in ["euclidean", "density"]] + ["ref"]:
            wanted |= {(str(s), int(x), int(y)) for s, (x, y) in zip(d[f"{key}_slides"].flat,
                                                                      d[f"{key}_xy"].reshape(-1, 2))}
    # 8 at a time, as GDC takes about 27 s to cut one tile from its slide
    with concurrent.futures.ThreadPoolExecutor(8) as pool:
        futures = {tile: pool.submit(tile_image, *tile) for tile in sorted(wanted)}
        images = {}
        for i, (tile, future) in enumerate(futures.items()):
            images[tile] = future.result()
            if (i + 1) % 50 == 0 or i + 1 == len(wanted):
                print(f"tiles {i + 1}/{len(wanted)}", flush=True)

    encoder = load_encoder(device)
    sae = TopKSAE(device)
    sae.load_state_dict(torch.load(sae_path, map_location=device))
    sae.eval().requires_grad_(False)
    classifier = TileClassifier()
    classifier.load_state_dict(torch.load(checkpoint, map_location="cpu"))
    head = classifier.head.eval().requires_grad_(False).to(device)
    v = (sae.decoder.weight.T @ head.weight[0]).cpu().numpy()

    # the reference tiles are the same in both npz files
    reference = {int(k): [(str(s), int(x), int(y)) for s, (x, y) in zip(slides, xy)]
                 for k, slides, xy in zip(data["l1"]["ref_concepts"], data["l1"]["ref_slides"], data["l1"]["ref_xy"])}
    inputs = {tile: model_input(images[tile]) for tiles in reference.values() for tile in tiles}
    # the crops must explain the pixels the model saw, so each re-embedded tile must match its stored vector
    with torch.inference_mode():
        cosines = [torch.cosine_similarity(encoder(tensor[None].to(device))[0].cpu(),
                                           torch.from_numpy(stored_vector(*tile)), dim=0).item()
                   for tile, (_, tensor) in inputs.items()]
    print(f"{len(inputs)} reference tiles re-embedded: cosine with the stored vector median {np.median(cosines):.4f}, "
          f"lowest {min(cosines):.4f}", flush=True)
    if min(cosines) < 0.99:
        raise ValueError(f"{sum(c < 0.99 for c in cosines)} reference tiles do not reproduce their stored vector")

    # step 9 may already have patched this kernel, and a second patch would wrap the forwards twice
    if not _patched:
        patch_for_lrp()
    composite = lrp_composite()
    parts = nn.ModuleDict({"encoder": encoder, "sae": sae, "head": head})
    crops = {}
    for n, (k, tiles) in enumerate(reference.items()):
        # the concept's own evidence: its contributions all have the sign of v_k, so the map for ILC for concepts that
        # push toward ILC and for NST (minus the logit) for the others
        crops[k] = [reference_heatmap_crop(inputs[tile][0], np.sign(v[k]) * concept_heatmap(
            parts, inputs[tile][1][None].to(device), k, composite)) for tile in tiles]
        print(f"concept {k}: maps {n + 1}/{len(reference)}", flush=True)

    for variant, d in data.items():
        for label in ["ILC", "NST"]:
            for mode in ["abs", "signed"]:
                columns = d[f"{label}_{mode}_columns"]
                concepts = [int(k) for k in d[f"{label}_{mode}_concepts"]]
                for select in ["euclidean", "density"]:
                    shown = [[images[str(s), int(x), int(y)] for s, (x, y) in zip(slides, xy)] for slides, xy in
                             zip(d[f"{label}_{select}_slides"][columns], d[f"{label}_{select}_xy"][columns])]
                    # prototype numbers as in the pcx_tiles report
                    plot_concept_prototype_matrix(shown, {k: crops[k] for k in concepts}, d[f"{label}_{mode}_matrix"],
                                                  concepts, matrix_path.format(variant, select, label, mode),
                                                  columns + 1, variant == "l1")


# the PDFs run_pcx_figure writes; Jupyter does not show PDFs inline, so they are listed
matrices = [matrix_path.format(variant, select, label, mode) for variant in ["raw", "l1"]
            for select in ["euclidean", "density"] for label in ["ILC", "NST"] for mode in ["abs", "signed"]]
load_or_run("pcx_figure", run_pcx_figure, matrices)